**Audit Extra Trees at Store × Department × Week**

Existing Extra Trees department results

In [4]:

import pandas as pd
import numpy as np
from google.colab import drive
from IPython.display import display

drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/Walmart Recruiting Sales Data"

dept = pd.read_csv(
    f"{BASE}/et_mad_department_week.csv",
    parse_dates=["Date"]
)

print("Dataset shape:", dept.shape)
print("Unique stores:", dept["Store"].nunique())
print("Unique departments:", dept["Dept"].nunique())
print("Store types:", dept["Type"].unique())
print("Date range:", dept["Date"].min(), "to",
      dept["Date"].max())

display(dept.head())


Mounted at /content/drive
Dataset shape: (42400, 13)
Unique stores: 45
Unique departments: 76
Store types: ['A' 'B' 'C']
Date range: 2012-07-20 00:00:00 to 2012-10-26 00:00:00


,Store,Dept,Date,Type,Actual_Sales,Expected_Sales,Dollar_Shortfall,Shortfall_Pct,Flag_GlobalMAD,Flag_TypeMAD,Flag_Fixed10,Flag_P90,Flag_P95
0,1,1,2012-07-20,A,16348.06,16635.950000,287.890000,1.730529,False,False,False,False,False
1,1,1,2012-07-27,A,15731.18,16478.871711,747.691711,4.537275,False,False,False,False,False
2,1,1,2012-08-03,A,16628.31,17104.066722,475.756722,2.781542,False,False,False,False,False
3,1,1,2012-08-10,A,16119.92,16973.876906,853.956906,5.031007,False,False,False,False,False
4,1,1,2012-08-17,A,17330.70,15789.795489,0.000000,0.000000,False,False,False,False,False


Verify calculations and MAD flags

In [5]:

# Verify observation uniqueness
keys = ["Store", "Dept", "Date"]

print("Duplicate department-weeks:",
      dept.duplicated(keys).sum())

assert not dept.duplicated(keys).any(), (
    "Duplicate Store-Dept-Date observations found."
)

# Convert the Global MAD flags to Boolean
dept["Flag_GlobalMAD"] = (
    dept["Flag_GlobalMAD"].astype(str).str.lower()
    .map({
        "true": True,
        "false": False,
        "1": True,
        "0": False
    })
)

assert dept["Flag_GlobalMAD"].notna().all()

# Recalculate residuals
dept["Residual_Check"] = (
    dept["Actual_Sales"] -
    dept["Expected_Sales"]
)

dept["Positive_Shortfall_Check"] = (
    -dept["Residual_Check"]
).clip(lower=0)

# Check the saved Dollar_Shortfall definition
dept["Raw_Shortfall_Check"] = (
    dept["Expected_Sales"] -
    dept["Actual_Sales"]
)

raw_diff = np.isclose(
    dept["Dollar_Shortfall"],
    dept["Raw_Shortfall_Check"],
    rtol=1e-6,
    atol=0.01
)

positive_diff = np.isclose(
    dept["Dollar_Shortfall"],
    dept["Positive_Shortfall_Check"],
    rtol=1e-6,
    atol=0.01
)

print("Matches signed shortfall:",
      round(raw_diff.mean() * 100, 2), "%")

print("Matches positive-only shortfall:",
      round(positive_diff.mean() * 100, 2), "%")

print("\nMissing values:")
display(dept.isna().sum())

print("\nGlobal MAD flag distribution:")
display(dept["Flag_GlobalMAD"].value_counts())


Duplicate department-weeks: 0
Matches signed shortfall: 58.93 %
Matches positive-only shortfall: 100.0 %

Missing values:


,0
Store,0
Dept,0
Date,0
Type,0
Actual_Sales,0
Expected_Sales,0
Dollar_Shortfall,0
Shortfall_Pct,0
Flag_GlobalMAD,0
Flag_TypeMAD,0



Global MAD flag distribution:


,count
Flag_GlobalMAD,
False,38035
True,4365


Evaluate department-level prediction accuracy

In [6]:

from sklearn.metrics import mean_absolute_error, mean_squared_error

valid = dept[
    dept["Actual_Sales"].notna() &
    dept["Expected_Sales"].notna()
].copy()

actual = valid["Actual_Sales"]
predicted = valid["Expected_Sales"]

mae = mean_absolute_error(actual, predicted)
rmse = np.sqrt(mean_squared_error(actual, predicted))

denominator = actual.abs().sum()

wape = (
    100 * (actual - predicted).abs().sum()
    / denominator
    if denominator > 0 else np.nan
)

bias = (predicted - actual).mean()

results = pd.DataFrame({
    "Metric": [
        "MAE",
        "RMSE",
        "WAPE (%)",
        "Mean Prediction Bias ($)"
    ],
    "Value": [mae, rmse, wape, bias]
})

display(results.round(2))

print("\nPrediction accuracy by store type:")

by_type = []

for store_type, group in valid.groupby("Type"):
    actual_sum = group["Actual_Sales"].abs().sum()

    by_type.append({
        "Store_Type": store_type,
        "Observations": len(group),
        "MAE": (
            group["Actual_Sales"] -
            group["Expected_Sales"]
        ).abs().mean(),
        "WAPE_%": (
            100 * (
                group["Actual_Sales"] -
                group["Expected_Sales"]
            ).abs().sum() / actual_sum
            if actual_sum > 0 else np.nan
        ),
        "Mean_Bias_$": (
            group["Expected_Sales"] -
            group["Actual_Sales"]
        ).mean()
    })

display(pd.DataFrame(by_type).round(2))


,Metric,Value
0,MAE,1280.25
1,RMSE,2630.58
2,WAPE (%),7.86
3,Mean Prediction Bias ($),174.03



Prediction accuracy by store type:


,Store_Type,Observations,MAE,WAPE_%,Mean_Bias_$
0,A,21791,1519.98,7.45,239.55
1,B,16424,1144.49,9.25,123.17
2,C,4185,564.78,5.45,32.53


Audit the flagged departments

In [7]:

dept["Shortfall_Positive"] = (
    dept["Expected_Sales"] >
    dept["Actual_Sales"]
)

dept_summary = (
    dept.groupby("Type")
    .agg(
        Department_Weeks=("Dept", "size"),
        Flagged_Dept_Weeks=("Flag_GlobalMAD", "sum"),
        Negative_Sales_Gaps=("Shortfall_Positive", "sum")
    )
    .reset_index()
)

dept_summary["Flag_Rate_%"] = (
    100 * dept_summary["Flagged_Dept_Weeks"]
    / dept_summary["Department_Weeks"]
)

display(dept_summary.round(2))

# Flagged departments without positive sales shortfalls
unexpected = dept[
    dept["Flag_GlobalMAD"] &
    ~dept["Shortfall_Positive"]
]

print("Flagged departments without positive shortfalls:",
      len(unexpected))

display(unexpected[
    ["Store", "Dept", "Date", "Type",
     "Actual_Sales", "Expected_Sales",
     "Dollar_Shortfall", "Shortfall_Pct"]
].head(10))


,Type,Department_Weeks,Flagged_Dept_Weeks,Negative_Sales_Gaps,Flag_Rate_%
0,A,21791,1932,12906,8.87
1,B,16424,1741,9564,10.60
2,C,4185,692,2517,16.54


Flagged departments without positive shortfalls: 0


,Store,Dept,Date,Type,Actual_Sales,Expected_Sales,Dollar_Shortfall,Shortfall_Pct


Aggregate department results to Store × Week

In [8]:

dept["Flagged_Dollar_Gap"] = np.where(
    dept["Flag_GlobalMAD"],
    dept["Positive_Shortfall_Check"],
    0
)

store_week = (
    dept.groupby(["Store", "Date", "Type"])
    .agg(
        Eligible_Dept_Count=("Dept", "nunique"),
        Flagged_Dept_Count=("Flag_GlobalMAD", "sum"),
        Expected_Sales=("Expected_Sales", "sum"),
        Actual_Sales=("Actual_Sales", "sum"),
        Gross_Dept_Shortfall=(
            "Positive_Shortfall_Check", "sum"
        ),
        Flagged_Dept_Shortfall=(
            "Flagged_Dollar_Gap", "sum"
        )
    )
    .reset_index()
)

store_week["Net_Sales_Gap"] = (
    store_week["Expected_Sales"] -
    store_week["Actual_Sales"]
)

store_week["Positive_Net_Shortfall"] = (
    store_week["Net_Sales_Gap"].clip(lower=0)
)

store_week["Flagged_Dept_Rate"] = (
    store_week["Flagged_Dept_Count"] /
    store_week["Eligible_Dept_Count"]
)

store_week["ET_Any_Dept"] = (
    store_week["Flagged_Dept_Count"] > 0
)

store_week["ET_10pct"] = (
    (store_week["Flagged_Dept_Rate"] >= 0.10) &
    (store_week["Net_Sales_Gap"] > 0)
)

store_week["ET_20pct"] = (
    (store_week["Flagged_Dept_Rate"] >= 0.20) &
    (store_week["Net_Sales_Gap"] > 0)
)

print("Store-weeks:", len(store_week))

display(
    store_week[
        ["ET_Any_Dept", "ET_10pct", "ET_20pct"]
    ].agg(["sum", "mean"]).round(3)
)

print("\nStore type breakdown:")

display(
    store_week.groupby("Type")[
        ["ET_Any_Dept", "ET_10pct", "ET_20pct"]
    ].agg(["sum", "mean"]).round(3)
)


Store-weeks: 675


,ET_Any_Dept,ET_10pct,ET_20pct
sum,672.000,251.000,41.000
mean,0.996,0.372,0.061



Store type breakdown:


ET_Any_Dept        ET_10pct        ET_20pct       
             sum   mean      sum   mean      sum   mean
Type                                                   
A            328  0.994      100  0.303       15  0.045
B            254  0.996      101  0.396       13  0.051
C             90  1.000       50  0.556       13  0.144

Find important exceptions missed by the 20% rule

In [9]:

import pandas as pd
import numpy as np

# Find the correct department count column
if "Dept_Count" in store_week.columns:
    count_col = "Dept_Count"
elif "Eligible_Dept_Count" in store_week.columns:
    count_col = "Eligible_Dept_Count"
else:
    raise KeyError("Department count column not found.")

# Store-weeks with positive shortfalls missed by 20% rule
missed = store_week[
    (store_week["Positive_Net_Shortfall"] > 0) &
    (~store_week["ET_20pct"])
].copy()

missed = missed.sort_values(
    "Positive_Net_Shortfall",
    ascending=False
)

print("TOP 10 POSITIVE NET SHORTFALLS")
print("NOT FLAGGED BY THE 20% RULE")

display(missed[
    ["Store", "Date", "Type",
     count_col,
     "Flagged_Dept_Count",
     "Flagged_Dept_Rate",
     "Positive_Net_Shortfall"]
].head(10))

# Compare modeled net shortfall coverage
total_shortfall = store_week[
    "Positive_Net_Shortfall"
].sum()

coverage = []

for method in ["ET_Any_Dept", "ET_10pct", "ET_20pct"]:
    captured = store_week.loc[
        store_week[method],
        "Positive_Net_Shortfall"
    ].sum()

    coverage.append({
        "Method": method,
        "Flagged_Weeks": int(store_week[method].sum()),
        "Shortfall_Coverage_%": (
            100 * captured / total_shortfall
            if total_shortfall > 0 else np.nan
        )
    })

print("\nESTIMATED NET SHORTFALL COVERAGE")
display(pd.DataFrame(coverage).round(2))


TOP 10 POSITIVE NET SHORTFALLS
NOT FLAGGED BY THE 20% RULE


,Store,Date,Type,Eligible_Dept_Count,Flagged_Dept_Count,Flagged_Dept_Rate,Positive_Net_Shortfall
398,27,2012-09-14,A,69,13,0.188406,240020.310072
188,13,2012-09-14,A,70,13,0.185714,187831.970798
391,27,2012-07-27,A,68,9,0.132353,164796.601894
203,14,2012-09-14,A,68,12,0.176471,159678.308767
141,10,2012-08-31,B,68,7,0.102941,156812.001294
578,39,2012-09-14,A,68,12,0.176471,155577.737017
338,23,2012-09-14,B,67,7,0.104478,143381.675617
181,13,2012-07-27,A,70,7,0.100000,142896.111567
350,24,2012-08-24,A,70,13,0.185714,125007.710839
256,18,2012-07-27,B,67,9,0.134328,124642.646400



ESTIMATED NET SHORTFALL COVERAGE


,Method,Flagged_Weeks,Shortfall_Coverage_%
0,ET_Any_Dept,672,99.92
1,ET_10pct,251,64.52
2,ET_20pct,41,11.31


Verify whether Type C is actually included

In [10]:

# Verify evaluation coverage for all store types

coverage = (
    store_week.groupby("Type")
    .agg(
        Stores=("Store", "nunique"),
        Store_Weeks=("Store", "size"),
        Positive_Net_Shortfall_Weeks=(
            "Positive_Net_Shortfall",
            lambda x: (x > 0).sum()
        )
    )
)

print("EVALUATION COVERAGE")
display(coverage)

assert set(coverage.index) == {"A", "B", "C"}
assert store_week[["Store", "Date"]].duplicated().sum() == 0


EVALUATION COVERAGE


,Stores,Store_Weeks,Positive_Net_Shortfall_Weeks
Type,,,
A,22,330,214
B,17,255,163
C,6,90,50


Chronological Isolation Forest (IF-A)

In [11]:

# CELL 8 — CHRONOLOGICAL ISOLATION FOREST (IF-A)

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from IPython.display import display

# Reuse BASE from Cell 1
base_path = Path(BASE)

train = pd.read_csv(base_path / "train.csv", parse_dates=["Date"])
features = pd.read_csv(base_path / "features.csv", parse_dates=["Date"])
stores = pd.read_csv(base_path / "stores.csv")

# Prevent duplicate rows during merging
assert not features.duplicated(["Store", "Date"]).any()
assert not stores.duplicated(["Store"]).any()

# Merge historical Walmart data
data = (
    train.merge(features, on=["Store", "Date", "IsHoliday"],
                how="left", validate="many_to_one")
         .merge(stores[["Store", "Type"]], on="Store",
                how="left", validate="many_to_one")
)

markdown_cols = [f"MarkDown{i}" for i in range(1, 6)]
data[markdown_cols] = data[markdown_cols].fillna(0)

# Aggregate to Store x Week
sales = (
    data.groupby(["Store", "Date"], as_index=False)
        ["Weekly_Sales"].sum()
)

context = data.drop_duplicates(["Store", "Date"])[
    ["Store", "Date", "Type"] + markdown_cols +
    ["Temperature", "Fuel_Price", "CPI", "Unemployment"]
]

sw = sales.merge(
    context, on=["Store", "Date"],
    validate="one_to_one"
)

sw["Total_MarkDown"] = sw[markdown_cols].sum(axis=1)

# Same features as original Isolation Forest
if_features = [
    "Weekly_Sales",
    "Total_MarkDown",
    "Temperature",
    "Fuel_Price",
    "CPI",
    "Unemployment"
]

# Chronological train/test split
cutoff = pd.Timestamp("2012-07-20")
end_date = pd.Timestamp("2012-10-26")

train_if = sw[sw["Date"] < cutoff].copy()
test_if = sw[
    sw["Date"].between(cutoff, end_date)
].copy()

# Ensure valid features without using test data
train_if = train_if.dropna(subset=if_features)
test_if = test_if.dropna(subset=if_features)

# Scale using historical training data only
scaler = StandardScaler()
X_train = scaler.fit_transform(train_if[if_features])
X_test = scaler.transform(test_if[if_features])

# Fit on historical observations only
iso = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)

iso.fit(X_train)

# Predict anomalies in held-out store-weeks
test_if["IF_Prediction"] = iso.predict(X_test)
test_if["IF_Flag"] = (test_if["IF_Prediction"] == -1)

# Lower decision scores mean more anomalous
test_if["IF_Score"] = iso.decision_function(X_test)

print("Training period:",
      train_if["Date"].min().date(), "to",
      train_if["Date"].max().date())

print("Evaluation period:",
      test_if["Date"].min().date(), "to",
      test_if["Date"].max().date())

print("Training rows:", len(train_if))
print("Evaluation rows:", len(test_if))

print("\nISOLATION FOREST FLAGS BY STORE TYPE")

if_summary = (
    test_if.groupby("Type")
    .agg(
        Stores=("Store", "nunique"),
        Store_Weeks=("Store", "size"),
        IF_Flags=("IF_Flag", "sum"),
        Mean_Score=("IF_Score", "mean"),
        Min_Score=("IF_Score", "min")
    )
)

if_summary["Flag_Rate_%"] = (
    100 * if_summary["IF_Flags"] /
    if_summary["Store_Weeks"]
)

display(if_summary.round(4))

print("\nTOTAL IF FLAGS:", test_if["IF_Flag"].sum())

# Check against the exact ET evaluation store-weeks
et_keys = store_week[["Store", "Date"]].drop_duplicates()
if_keys = test_if[["Store", "Date"]].drop_duplicates()

matches = et_keys.merge(
    if_keys, on=["Store", "Date"], how="outer",
    indicator=True
)

print("\nET/IF evaluation key matching:")
print(matches["_merge"].value_counts())

# Save for later comparison
test_if.to_csv(
    base_path / "audit_if_chronological.csv",
    index=False
)


Training period: 2010-02-05 to 2012-07-13
Evaluation period: 2012-07-20 to 2012-10-26
Training rows: 5760
Evaluation rows: 675

ISOLATION FOREST FLAGS BY STORE TYPE


,Stores,Store_Weeks,IF_Flags,Mean_Score,Min_Score,Flag_Rate_%
Type,,,,,,
A,22,330,50,0.0386,-0.0969,15.1515
B,17,255,19,0.0539,-0.0605,7.4510
C,6,90,0,0.0694,0.0055,0.0000



TOTAL IF FLAGS: 69

ET/IF evaluation key matching:
_merge
both          675
left_only       0
right_only      0
Name: count, dtype: int64


Check available historical predictions

In [12]:

# CELL 9 — CHECK HISTORICAL EXTRA TREES PREDICTIONS

import os
import pandas as pd
from pathlib import Path

base_path = Path(BASE)

# Find possible historical prediction files
csv_files = list(base_path.rglob("*.csv"))

print("AVAILABLE CSV FILES")
for file in csv_files:
    print(file.name)

print("\nCHECKING EXTRA TREES HISTORICAL DATA")

# Examine likely model-output files
for file in csv_files:
    if any(word in file.name.lower()
           for word in ["et_", "prediction", "forecast",
                        "residual", "extra"]):
        try:
            sample = pd.read_csv(file, nrows=5)
            print(f"\nFile: {file.name}")
            print("Columns:", sample.columns.tolist())
        except Exception as e:
            print("Could not read:", file.name, e)

# Check date coverage of our current ET department export
print("\nCURRENT EXTRA TREES DEPARTMENT EXPORT")

print("Earliest date:", dept["Date"].min())
print("Latest date:", dept["Date"].max())

print("Rows:", len(dept))

print("\nHistorical predictions before 2012-07-20:",
      (dept["Date"] < "2012-07-20").sum())


AVAILABLE CSV FILES
test.csv
train.csv
sampleSubmission.csv
features.csv
stores.csv
isolation_forest_store_week.csv
isolation_forest_all_45_results (2).csv
et_mad_store_week.csv
et_mad_department_week.csv
additional_benchmarks_validation.csv
additional_benchmarks_final_test.csv
additional_benchmarks_predictions.csv
combined_updated_with_benchmarks.csv
combined_evaluation_matrix.csv
FINAL_hybrid_top20_cases.csv
rag_cases_v3_summary.csv
rag_cases_v3_1_summary.csv
rag_cases_v3_2_validation_summary.csv
if_original_vs_chronological_summary.csv
if_original_vs_chronological_comparison.csv
audit_if_chronological.csv
audit_historical_ma4_residuals.csv
audit_if_residual_comparison.csv
audit_et_if_business_comparison.csv
audit_equal_budget_top20.csv
audit_hybrid_top20_comparison.csv
audit_forecast_model_comparison.csv
audit_forecast_by_store_type.csv
existing_forecast_model_ranking.csv
existing_forecast_by_store_type.csv
existing_forecast_predictions_matched.csv
gradient_boosting_final_results.cs

Check benchmark prediction coverage

In [13]:

# CELL 10 — CHECK EXISTING BENCHMARK PREDICTIONS

import pandas as pd

benchmark_path = base_path / "additional_benchmarks_predictions.csv"

bench = pd.read_csv(
    benchmark_path,
    parse_dates=["Date"]
)

print("DATASET SHAPE:", bench.shape)
print("DATE RANGE:",
      bench["Date"].min().date(),
      "to",
      bench["Date"].max().date())

print("\nAVAILABLE MODELS")
print(bench["Model"].value_counts())

print("\nFIRST FEW ROWS")
display(bench.head())

print("\nHISTORICAL PREDICTIONS BEFORE JULY 20, 2012")
historical = bench[
    bench["Date"] < "2012-07-20"
]

print("Historical prediction rows:", len(historical))

if len(historical) > 0:
    print("\nHistorical models:")
    print(historical["Model"].value_counts())
else:
    print("No historical predictions available in this file.")

print("\nEVALUATION-PERIOD PREDICTIONS")
evaluation = bench[
    bench["Date"].between("2012-07-20", "2012-10-26")
]

print("Evaluation rows:", len(evaluation))
print(evaluation["Model"].value_counts())


DATASET SHAPE: (168208, 6)
DATE RANGE: 2012-07-20 to 2012-10-26

AVAILABLE MODELS
Model
MA4                     42052
MA8                     42052
MA12                    42052
HistGradientBoosting    42052
Name: count, dtype: int64

FIRST FEW ROWS


,Store,Dept,Date,Weekly_Sales,Model,Prediction
0,1,1,2012-07-20,16348.06,MA4,16686.4450
1,1,1,2012-07-27,15731.18,MA4,16615.1075
2,1,1,2012-08-03,16628.31,MA4,16617.1975
3,1,1,2012-08-10,16119.92,MA4,16318.4325
4,1,1,2012-08-17,17330.70,MA4,16206.8675



HISTORICAL PREDICTIONS BEFORE JULY 20, 2012
Historical prediction rows: 0
No historical predictions available in this file.

EVALUATION-PERIOD PREDICTIONS
Evaluation rows: 168208
Model
MA4                     42052
MA8                     42052
MA12                    42052
HistGradientBoosting    42052
Name: count, dtype: int64


Generate historical residuals

In [14]:

# CELL 11 — HISTORICAL 4-WEEK MOVING AVERAGE RESIDUALS

import pandas as pd
import numpy as np

# sw was created in Cell 8 and contains all historical store-weeks
sw_resid = sw.sort_values(["Store", "Date"]).copy()

# Lagged 4-week average, using prior observations only
sw_resid["MA4_Expected"] = (
    sw_resid.groupby("Store")["Weekly_Sales"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=4, min_periods=4
        ).mean()
    )
)

# Actual minus historical expectation
sw_resid["MA4_Residual"] = (
    sw_resid["Weekly_Sales"] -
    sw_resid["MA4_Expected"]
)

# Relative residual for comparison across differently sized stores
sw_resid["MA4_Residual_Pct"] = np.where(
    sw_resid["MA4_Expected"] > 0,
    sw_resid["MA4_Residual"] /
    sw_resid["MA4_Expected"],
    np.nan
)

# Remove initial weeks without four prior observations
sw_resid = sw_resid.dropna(
    subset=["MA4_Expected", "MA4_Residual_Pct"]
).copy()

cutoff = pd.Timestamp("2012-07-20")
end_date = pd.Timestamp("2012-10-26")

train_resid = sw_resid[
    sw_resid["Date"] < cutoff
].copy()

test_resid = sw_resid[
    sw_resid["Date"].between(cutoff, end_date)
].copy()

print("HISTORICAL RESIDUAL DATASET")

print("Training rows:", len(train_resid))
print("Evaluation rows:", len(test_resid))

print("\nTRAINING COVERAGE BY STORE TYPE")
display(
    train_resid.groupby("Type")
    .agg(
        Stores=("Store", "nunique"),
        Store_Weeks=("Store", "size")
    )
)

print("\nEVALUATION COVERAGE BY STORE TYPE")
display(
    test_resid.groupby("Type")
    .agg(
        Stores=("Store", "nunique"),
        Store_Weeks=("Store", "size")
    )
)

# Verify same evaluation keys as IF-A
matching = test_resid[["Store", "Date"]].merge(
    test_if[["Store", "Date"]],
    on=["Store", "Date"],
    how="outer",
    indicator=True,
    validate="one_to_one"
)

print("\nIF-A / IF-B EVALUATION KEY MATCHING")
print(matching["_merge"].value_counts())

# Save historical residuals
sw_resid.to_csv(
    base_path / "audit_historical_ma4_residuals.csv",
    index=False
)


HISTORICAL RESIDUAL DATASET
Training rows: 5580
Evaluation rows: 675

TRAINING COVERAGE BY STORE TYPE


,Stores,Store_Weeks
Type,,
A,22,2728
B,17,2108
C,6,744



EVALUATION COVERAGE BY STORE TYPE


,Stores,Store_Weeks
Type,,
A,22,330
B,17,255
C,6,90



IF-A / IF-B EVALUATION KEY MATCHING
_merge
both          675
left_only       0
right_only      0
Name: count, dtype: int64


In [15]:

# CELL 12 — CONTROLLED ISOLATION FOREST COMPARISON

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

original_features = [
    "Weekly_Sales",
    "Total_MarkDown",
    "Temperature",
    "Fuel_Price",
    "CPI",
    "Unemployment"
]

residual_features = original_features + [
    "MA4_Residual_Pct"
]

# Same eligible observations for both experiments
train_compare = train_resid.dropna(
    subset=residual_features
).copy()

test_compare = test_resid.dropna(
    subset=residual_features
).copy()

assert len(test_compare) == 675, (
    "Expected 675 evaluation store-weeks."
)
assert not test_compare.duplicated(
    ["Store", "Date"]
).any()

results = test_compare[
    ["Store", "Date", "Type", "Weekly_Sales",
     "MA4_Residual_Pct"]
].copy()

for name, cols in {
    "IF_A": original_features,
    "IF_B": residual_features
}.items():

    scaler = StandardScaler()

    X_train = scaler.fit_transform(
        train_compare[cols]
    )

    X_test = scaler.transform(
        test_compare[cols]
    )

    model = IsolationForest(
        n_estimators=200,
        contamination=0.05,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train)

    results[f"{name}_Flag"] = (
        model.predict(X_test) == -1
    )

    results[f"{name}_Score"] = (
        model.decision_function(X_test)
    )

# Compare anomaly counts
print("OVERALL COMPARISON")

summary = []

for name in ["IF_A", "IF_B"]:
    summary.append({
        "Model": name,
        "Flagged_Weeks": int(
            results[f"{name}_Flag"].sum()
        ),
        "Flag_Rate_%": round(
            results[f"{name}_Flag"].mean() * 100, 2
        )
    })

display(pd.DataFrame(summary))

# Breakdown by store type
print("\nFLAGS BY STORE TYPE")

display(
    results.groupby("Type")[
        ["IF_A_Flag", "IF_B_Flag"]
    ].agg(["sum", "mean"]).round(3)
)

# Model agreement
both = (
    results["IF_A_Flag"] &
    results["IF_B_Flag"]
).sum()

either = (
    results["IF_A_Flag"] |
    results["IF_B_Flag"]
).sum()

print("\nMODEL AGREEMENT")
print("Both flag:", both)
print("Only IF-A:", int(
    (results["IF_A_Flag"] &
     ~results["IF_B_Flag"]).sum()
))
print("Only IF-B:", int(
    (~results["IF_A_Flag"] &
     results["IF_B_Flag"]).sum()
))
print("Jaccard similarity:",
      round(both / either, 3) if either else np.nan)

# Store Type C analysis
print("\nTYPE C ANOMALY SCORES")

display(
    results[results["Type"] == "C"][
        ["Store", "Date", "MA4_Residual_Pct",
         "IF_A_Score", "IF_B_Score",
         "IF_A_Flag", "IF_B_Flag"]
    ].sort_values("IF_B_Score").head(10)
)

results.to_csv(
    base_path / "audit_if_residual_comparison.csv",
    index=False
)

print("\nComparison saved.")


OVERALL COMPARISON


,Model,Flagged_Weeks,Flag_Rate_%
0,IF_A,64,9.48
1,IF_B,31,4.59



FLAGS BY STORE TYPE


IF_A_Flag        IF_B_Flag       
           sum   mean       sum   mean
Type                                  
A           47  0.142        25  0.076
B           16  0.063         6  0.024
C            1  0.011         0  0.000


MODEL AGREEMENT
Both flag: 30
Only IF-A: 34
Only IF-B: 1
Jaccard similarity: 0.462

TYPE C ANOMALY SCORES


,Store,Date,MA4_Residual_Pct,IF_A_Score,IF_B_Score,IF_A_Flag,IF_B_Flag
5426,38,2012-09-07,0.173686,0.003391,0.010857,False,False
5423,38,2012-08-17,-0.061204,-0.000446,0.029029,True,False
5425,38,2012-08-31,-0.009139,0.003403,0.041423,False,False
5422,38,2012-08-10,0.007232,0.016049,0.042637,False,False
5427,38,2012-09-14,-0.000404,0.009174,0.045188,False,False
5420,38,2012-07-27,-0.060397,0.026422,0.048593,False,False
5424,38,2012-08-24,-0.078456,0.024529,0.053759,False,False
5431,38,2012-10-12,0.030510,0.031325,0.054721,False,False
5428,38,2012-09-21,-0.060307,0.011579,0.054828,False,False
5432,38,2012-10-19,0.006665,0.037905,0.055037,False,False



Comparison saved.


Compare models and business impact

In [16]:

# CELL 13 — BUSINESS IMPACT COMPARISON
# Uses:
# store_week from Cell 5
# results from Cell 12

import pandas as pd
import numpy as np
from IPython.display import display

# 1. Join IF-A and IF-B with Extra Trees
comparison = store_week.merge(
    results[
        ["Store", "Date", "IF_A_Flag", "IF_B_Flag",
         "IF_A_Score", "IF_B_Score",
         "MA4_Residual_Pct"]
    ],
    on=["Store", "Date"],
    how="inner",
    validate="one_to_one"
)

assert len(comparison) == 675
assert not comparison.duplicated(["Store", "Date"]).any()

print("Matched store-weeks:", len(comparison))

# 2. Compare all approaches
methods = [
    "ET_10pct",
    "ET_20pct",
    "IF_A_Flag",
    "IF_B_Flag"
]

total_gap = comparison["Positive_Net_Shortfall"].sum()

summary = []

for method in methods:
    selected = comparison[comparison[method]]

    captured = selected["Positive_Net_Shortfall"].sum()

    summary.append({
        "Method": method,
        "Flagged_Weeks": len(selected),
        "Flag_Rate_%": (
            100 * len(selected) / len(comparison)
        ),
        "Captured_Net_Gap_$": captured,
        "Net_Gap_Coverage_%": (
            100 * captured / total_gap
            if total_gap > 0 else np.nan
        ),
        "Mean_Net_Gap_Per_Flag_$": (
            selected["Positive_Net_Shortfall"].mean()
            if len(selected) else np.nan
        )
    })

print("\nMODEL BUSINESS IMPACT COMPARISON")
display(pd.DataFrame(summary).round(2))

# 3. Store type breakdown
print("\nFLAGS BY STORE TYPE")

type_summary = (
    comparison.groupby("Type")[methods]
    .sum()
)

display(type_summary)

# 4. Analyze the 34 IF-A flags removed by IF-B
removed = comparison[
    comparison["IF_A_Flag"] &
    ~comparison["IF_B_Flag"]
].copy()

removed = removed.sort_values(
    "Positive_Net_Shortfall",
    ascending=False
)

print("\nIF-A CASES REMOVED BY IF-B")
print("Number removed:", len(removed))

print(
    "Total modeled positive net gap in removed cases: $",
    round(removed["Positive_Net_Shortfall"].sum(), 2)
)

display(removed[
    ["Store", "Date", "Type",
     "Positive_Net_Shortfall",
     "Flagged_Dept_Count",
     "Flagged_Dept_Rate",
     "MA4_Residual_Pct",
     "IF_A_Score", "IF_B_Score"]
].head(15))

# 5. Cases newly selected by IF-B
added = comparison[
    ~comparison["IF_A_Flag"] &
    comparison["IF_B_Flag"]
].copy()

print("\nNEW CASES INTRODUCED BY IF-B")

display(added[
    ["Store", "Date", "Type",
     "Positive_Net_Shortfall",
     "MA4_Residual_Pct",
     "IF_A_Score", "IF_B_Score"]
])

# 6. Save comparison
comparison.to_csv(
    base_path / "audit_et_if_business_comparison.csv",
    index=False
)

print("\nBusiness comparison saved.")


Matched store-weeks: 675

MODEL BUSINESS IMPACT COMPARISON


,Method,Flagged_Weeks,Flag_Rate_%,Captured_Net_Gap_$,Net_Gap_Coverage_%,Mean_Net_Gap_Per_Flag_$
0,ET_10pct,251,37.19,9877966.89,64.52,39354.45
1,ET_20pct,41,6.07,1730904.35,11.31,42217.18
2,IF_A_Flag,64,9.48,2121513.55,13.86,33148.65
3,IF_B_Flag,31,4.59,1190868.39,7.78,38415.11



FLAGS BY STORE TYPE


,ET_10pct,ET_20pct,IF_A_Flag,IF_B_Flag
Type,,,,
A,100,15,47,25
B,101,13,16,6
C,50,13,1,0



IF-A CASES REMOVED BY IF-B
Number removed: 34
Total modeled positive net gap in removed cases: $ 1074026.83


,Store,Date,Type,Positive_Net_Shortfall,Flagged_Dept_Count,Flagged_Dept_Rate,MA4_Residual_Pct,IF_A_Score,IF_B_Score
141,10,2012-08-31,B,156812.001294,7,0.102941,-0.109477,-0.022005,0.007583
577,39,2012-09-07,A,112593.702039,10,0.147059,-0.052729,-0.009802,0.034272
76,6,2012-07-27,A,110376.845739,5,0.071429,-0.094922,-0.008022,0.004147
201,14,2012-08-31,A,82389.207511,7,0.102941,-0.020396,-0.013879,0.037256
172,12,2012-09-07,B,79936.806244,11,0.169231,-0.068091,-0.020059,0.007592
351,24,2012-08-31,A,70325.160078,7,0.098592,-0.046264,-0.005816,0.030524
456,31,2012-08-31,A,64757.625026,9,0.130435,-0.012935,-0.024906,0.018513
156,11,2012-08-31,A,57334.781294,9,0.130435,-0.022717,-0.004532,0.033872
291,20,2012-08-31,A,54491.332439,4,0.057971,-0.004734,-0.003368,0.039542
166,12,2012-07-27,B,53829.554894,3,0.045455,-0.049746,-0.003267,0.019586



NEW CASES INTRODUCED BY IF-B


,Store,Date,Type,Positive_Net_Shortfall,MA4_Residual_Pct,IF_A_Score,IF_B_Score
338,23,2012-09-14,B,143381.675617,-0.177066,0.02669,-0.005052



Business comparison saved.


Equal-Budget Model Comparison

In [17]:

# CELL 14 — EQUAL-BUDGET MODEL COMPARISON

import pandas as pd
import numpy as np
from IPython.display import display

df = comparison.copy()

BUDGET = 20

# Higher priority = more anomalous for IF
df["IF_A_Priority"] = -df["IF_A_Score"]
df["IF_B_Priority"] = -df["IF_B_Score"]

methods = {
    "ET_10pct": ("ET_10pct", "Positive_Net_Shortfall"),
    "ET_20pct": ("ET_20pct", "Positive_Net_Shortfall"),
    "IF_A": ("IF_A_Flag", "IF_A_Priority"),
    "IF_B": ("IF_B_Flag", "IF_B_Priority")
}

selected_cases = {}
summary = []

for method, (flag, priority) in methods.items():

    selected = (
        df[df[flag]]
        .sort_values(priority, ascending=False)
        .head(BUDGET)
        .copy()
    )

    selected_cases[method] = selected

    summary.append({
        "Method": method,
        "Selected": len(selected),
        "Total_Net_Gap_$":
            selected["Positive_Net_Shortfall"].sum(),
        "Mean_Net_Gap_$":
            selected["Positive_Net_Shortfall"].mean(),
        "Type_A": (selected["Type"] == "A").sum(),
        "Type_B": (selected["Type"] == "B").sum(),
        "Type_C": (selected["Type"] == "C").sum()
    })

print("TOP-20 COMPARISON")
display(pd.DataFrame(summary).round(2))

# Show model-specific selected cases
for method, selected in selected_cases.items():
    print(f"\n{method} — TOP {len(selected)} CASES")

    display(selected[
        ["Store", "Date", "Type",
         "Positive_Net_Shortfall",
         "Flagged_Dept_Count",
         "Flagged_Dept_Rate",
         "IF_A_Score", "IF_B_Score"]
    ].head(10))

# Save selection table
top20_export = pd.concat(
    [part.assign(Method=name)
     for name, part in selected_cases.items()],
    ignore_index=True
)

top20_export.to_csv(
    base_path / "audit_equal_budget_top20.csv",
    index=False
)

print("\nEqual-budget comparison saved.")


TOP-20 COMPARISON


,Method,Selected,Total_Net_Gap_$,Mean_Net_Gap_$,Type_A,Type_B,Type_C
0,ET_10pct,20,2913592.41,145679.62,15,5,0
1,ET_20pct,20,1467462.30,73373.12,8,10,2
2,IF_A,20,465059.08,23252.95,15,5,0
3,IF_B,20,779674.01,38983.70,16,4,0



ET_10pct — TOP 20 CASES


,Store,Date,Type,Positive_Net_Shortfall,Flagged_Dept_Count,Flagged_Dept_Rate,IF_A_Score,IF_B_Score
398,27,2012-09-14,A,240020.310072,13,0.188406,0.045444,0.029693
188,13,2012-09-14,A,187831.970798,13,0.185714,0.040389,0.040100
337,23,2012-09-07,B,170762.843450,14,0.208955,0.008178,0.025937
83,6,2012-09-14,A,165278.405639,14,0.200000,0.003331,0.016465
391,27,2012-07-27,A,164796.601894,9,0.132353,0.050499,0.051748
203,14,2012-09-14,A,159678.308767,12,0.176471,0.052021,0.070566
141,10,2012-08-31,B,156812.001294,7,0.102941,-0.022005,0.007583
578,39,2012-09-14,A,155577.737017,12,0.176471,0.025089,0.032065
338,23,2012-09-14,B,143381.675617,7,0.104478,0.026690,-0.005052
181,13,2012-07-27,A,142896.111567,7,0.100000,0.021842,0.034433



ET_20pct — TOP 20 CASES


,Store,Date,Type,Positive_Net_Shortfall,Flagged_Dept_Count,Flagged_Dept_Rate,IF_A_Score,IF_B_Score
337,23,2012-09-07,B,170762.843450,14,0.208955,0.008178,0.025937
83,6,2012-09-14,A,165278.405639,14,0.200000,0.003331,0.016465
353,24,2012-09-14,A,140934.939459,17,0.239437,0.077534,0.071448
358,24,2012-10-19,A,127588.963733,15,0.211268,0.079233,0.074098
307,21,2012-09-07,B,121358.432861,19,0.301587,0.047184,0.027737
593,40,2012-09-14,A,106307.328921,16,0.235294,0.027658,0.022063
424,29,2012-08-17,B,99033.450656,23,0.370968,0.079026,0.064181
8,1,2012-09-14,A,80953.334063,14,0.200000,0.034419,0.066665
519,35,2012-09-21,B,73120.440283,13,0.209677,0.070126,0.033352
52,4,2012-09-07,A,53619.196883,16,0.228571,-0.024207,-0.018852



IF_A — TOP 20 CASES


,Store,Date,Type,Positive_Net_Shortfall,Flagged_Dept_Count,Flagged_Dept_Rate,IF_A_Score,IF_B_Score
47,4,2012-08-03,A,46637.280617,6,0.086957,-0.100413,-0.076488
105,8,2012-07-20,A,0.000000,5,0.075758,-0.093472,-0.039582
182,13,2012-08-03,A,58663.680883,5,0.072464,-0.079937,-0.049330
167,12,2012-08-03,B,25416.060878,6,0.090909,-0.071315,-0.041681
302,21,2012-08-03,B,30042.481178,12,0.184615,-0.062341,-0.036871
570,39,2012-07-20,A,0.000000,3,0.045455,-0.059953,-0.022005
171,12,2012-08-31,B,0.000000,5,0.075758,-0.056499,-0.024259
411,28,2012-08-31,A,113152.165367,11,0.159420,-0.056477,-0.020244
77,6,2012-08-03,A,48798.392889,7,0.101449,-0.055995,-0.037737
602,41,2012-08-03,A,0.000000,5,0.073529,-0.052453,-0.001700



IF_B — TOP 20 CASES


,Store,Date,Type,Positive_Net_Shortfall,Flagged_Dept_Count,Flagged_Dept_Rate,IF_A_Score,IF_B_Score
47,4,2012-08-03,A,46637.280617,6,0.086957,-0.100413,-0.076488
182,13,2012-08-03,A,58663.680883,5,0.072464,-0.079937,-0.049330
167,12,2012-08-03,B,25416.060878,6,0.090909,-0.071315,-0.041681
105,8,2012-07-20,A,0.000000,5,0.075758,-0.093472,-0.039582
77,6,2012-08-03,A,48798.392889,7,0.101449,-0.055995,-0.037737
302,21,2012-08-03,B,30042.481178,12,0.184615,-0.062341,-0.036871
17,2,2012-08-03,A,0.000000,2,0.028986,-0.049491,-0.034371
171,12,2012-08-31,B,0.000000,5,0.075758,-0.056499,-0.024259
48,4,2012-08-10,A,0.000000,5,0.072464,-0.040869,-0.024241
570,39,2012-07-20,A,0.000000,3,0.045455,-0.059953,-0.022005



Equal-budget comparison saved.


Extra Trees vs. Hybrid Model Evaluation

In [18]:

# CELL 15 — STANDALONE VS HYBRID EVALUATION

import pandas as pd
import numpy as np
from IPython.display import display

df = comparison.copy()

assert len(df) == 675
assert not df.duplicated(["Store", "Date"]).any()

# -----------------------------------
# 1. Define candidate approaches
# -----------------------------------

df["Hybrid_A_Flag"] = (
    df["ET_10pct"] & df["IF_A_Flag"]
)

df["Hybrid_B_Flag"] = (
    df["ET_10pct"] & df["IF_B_Flag"]
)

# Business-first ranking:
# Keep ET-eligible weeks and prioritize
# those also flagged by IF-A.
# Within each group, rank by ET net shortfall.

approaches = {
    "ET_Standalone": "ET_10pct",
    "Hybrid_A_Strict": "Hybrid_A_Flag",
    "Hybrid_B_Strict": "Hybrid_B_Flag",
    "Hybrid_C_BusinessFirst": "ET_10pct"
}

BUDGET = 20

selected_cases = {}
summary = []

total_gap = df["Positive_Net_Shortfall"].sum()

for name, flag_col in approaches.items():

    eligible = df[df[flag_col]].copy()

    if name == "Hybrid_C_BusinessFirst":
        eligible = eligible.sort_values(
            ["IF_A_Flag", "Positive_Net_Shortfall"],
            ascending=[False, False]
        )
    else:
        eligible = eligible.sort_values(
            "Positive_Net_Shortfall",
            ascending=False
        )

    selected = eligible.head(BUDGET).copy()

    selected_cases[name] = selected

    captured = selected["Positive_Net_Shortfall"].sum()

    summary.append({
        "Approach": name,
        "Eligible_Weeks": len(eligible),
        "Top20_Selected": len(selected),
        "Top20_Net_Gap_$": captured,
        "Top20_Coverage_%": (
            100 * captured / total_gap
            if total_gap > 0 else np.nan
        ),
        "Type_A": (selected["Type"] == "A").sum(),
        "Type_B": (selected["Type"] == "B").sum(),
        "Type_C": (selected["Type"] == "C").sum()
    })

print("STANDALONE VS HYBRID TOP-20 COMPARISON")
display(pd.DataFrame(summary).round(2))

# -----------------------------------
# 2. Compare hybrid against standalone
# -----------------------------------

et_top20 = set(
    zip(
        selected_cases["ET_Standalone"]["Store"],
        selected_cases["ET_Standalone"]["Date"]
    )
)

for name in [
    "Hybrid_A_Strict",
    "Hybrid_B_Strict",
    "Hybrid_C_BusinessFirst"
]:
    hybrid_top20 = set(
        zip(
            selected_cases[name]["Store"],
            selected_cases[name]["Date"]
        )
    )

    print(f"\n{name}")
    print("Overlap with ET top 20:",
          len(et_top20 & hybrid_top20))
    print("ET top-20 cases excluded:",
          len(et_top20 - hybrid_top20))

# -----------------------------------
# 3. Examine high-impact ET cases
#    excluded by the strict hybrids
# -----------------------------------

for name, flag_col in [
    ("Hybrid_A_Strict", "Hybrid_A_Flag"),
    ("Hybrid_B_Strict", "Hybrid_B_Flag")
]:

    missed = df[
        df["ET_10pct"] & ~df[flag_col]
    ].sort_values(
        "Positive_Net_Shortfall",
        ascending=False
    )

    print(f"\n{name}: TOP EXCLUDED ET CASES")

    display(missed[
        ["Store", "Date", "Type",
         "Positive_Net_Shortfall",
         "Flagged_Dept_Count",
         "Flagged_Dept_Rate",
         "IF_A_Flag", "IF_B_Flag"]
    ].head(10))

# -----------------------------------
# 4. Save results
# -----------------------------------

hybrid_export = pd.concat(
    [
        selected.assign(Approach=name)
        for name, selected in selected_cases.items()
    ],
    ignore_index=True
)

hybrid_export.to_csv(
    base_path / "audit_hybrid_top20_comparison.csv",
    index=False
)

print("\nHybrid evaluation saved.")


STANDALONE VS HYBRID TOP-20 COMPARISON


,Approach,Eligible_Weeks,Top20_Selected,Top20_Net_Gap_$,Top20_Coverage_%,Type_A,Type_B,Type_C
0,ET_Standalone,251,20,2913592.41,19.03,15,5,0
1,Hybrid_A_Strict,22,20,1270659.69,8.30,15,4,1
2,Hybrid_B_Strict,11,11,784612.47,5.12,9,2,0
3,Hybrid_C_BusinessFirst,251,20,1270659.69,8.30,15,4,1



Hybrid_A_Strict
Overlap with ET top 20: 4
ET top-20 cases excluded: 16

Hybrid_B_Strict
Overlap with ET top 20: 3
ET top-20 cases excluded: 17

Hybrid_C_BusinessFirst
Overlap with ET top 20: 4
ET top-20 cases excluded: 16

Hybrid_A_Strict: TOP EXCLUDED ET CASES


,Store,Date,Type,Positive_Net_Shortfall,Flagged_Dept_Count,Flagged_Dept_Rate,IF_A_Flag,IF_B_Flag
398,27,2012-09-14,A,240020.310072,13,0.188406,False,False
188,13,2012-09-14,A,187831.970798,13,0.185714,False,False
337,23,2012-09-07,B,170762.843450,14,0.208955,False,False
83,6,2012-09-14,A,165278.405639,14,0.200000,False,False
391,27,2012-07-27,A,164796.601894,9,0.132353,False,False
203,14,2012-09-14,A,159678.308767,12,0.176471,False,False
578,39,2012-09-14,A,155577.737017,12,0.176471,False,False
338,23,2012-09-14,B,143381.675617,7,0.104478,False,True
181,13,2012-07-27,A,142896.111567,7,0.100000,False,False
353,24,2012-09-14,A,140934.939459,17,0.239437,False,False



Hybrid_B_Strict: TOP EXCLUDED ET CASES


,Store,Date,Type,Positive_Net_Shortfall,Flagged_Dept_Count,Flagged_Dept_Rate,IF_A_Flag,IF_B_Flag
398,27,2012-09-14,A,240020.310072,13,0.188406,False,False
188,13,2012-09-14,A,187831.970798,13,0.185714,False,False
337,23,2012-09-07,B,170762.843450,14,0.208955,False,False
83,6,2012-09-14,A,165278.405639,14,0.200000,False,False
391,27,2012-07-27,A,164796.601894,9,0.132353,False,False
203,14,2012-09-14,A,159678.308767,12,0.176471,False,False
141,10,2012-08-31,B,156812.001294,7,0.102941,True,False
578,39,2012-09-14,A,155577.737017,12,0.176471,False,False
181,13,2012-07-27,A,142896.111567,7,0.100000,False,False
353,24,2012-09-14,A,140934.939459,17,0.239437,False,False



Hybrid evaluation saved.


Extra Trees vs. MA4, MA8, MA12, and Gradient Boosting

In [19]:

# CELL 16 — EXTRA TREES VS FORECASTING BASELINES

import pandas as pd
import numpy as np
from IPython.display import display

# Existing benchmark predictions
bench = pd.read_csv(
    base_path / "additional_benchmarks_predictions.csv",
    parse_dates=["Date"]
)

# Extra Trees predictions from our department audit
et = dept[
    ["Store", "Dept", "Date",
     "Actual_Sales", "Expected_Sales", "Type"]
].copy()

et = et.rename(columns={
    "Expected_Sales": "Prediction"
})
et["Model"] = "ExtraTrees"

# Standardize benchmark columns
bench = bench.rename(columns={
    "Weekly_Sales": "Actual_Sales"
})

# Add store type
bench = bench.merge(
    stores[["Store", "Type"]],
    on="Store",
    how="left",
    validate="many_to_one"
)

# Combine all models
all_predictions = pd.concat(
    [
        et[["Store", "Dept", "Date", "Type",
            "Actual_Sales", "Prediction", "Model"]],
        bench[["Store", "Dept", "Date", "Type",
               "Actual_Sales", "Prediction", "Model"]]
    ],
    ignore_index=True
)

# Keep only observations available in every model
keys = ["Store", "Dept", "Date"]

model_count = all_predictions["Model"].nunique()

common_keys = (
    all_predictions.groupby(keys)["Model"]
    .nunique()
    .reset_index(name="Model_Count")
)

common_keys = common_keys[
    common_keys["Model_Count"] == model_count
][keys]

matched = all_predictions.merge(
    common_keys,
    on=keys,
    how="inner",
    validate="many_to_one"
)

print("Common department-week observations:",
      len(common_keys))

print("Models compared:",
      matched["Model"].nunique())

# Calculate forecast errors
matched["Error"] = (
    matched["Prediction"] -
    matched["Actual_Sales"]
)

matched["Absolute_Error"] = matched["Error"].abs()
matched["Squared_Error"] = matched["Error"] ** 2

def evaluate(group):
    denom = group["Actual_Sales"].abs().sum()

    return pd.Series({
        "Observations": len(group),
        "MAE_$": group["Absolute_Error"].mean(),
        "RMSE_$": np.sqrt(
            group["Squared_Error"].mean()
        ),
        "WAPE_%": (
            100 * group["Absolute_Error"].sum() / denom
            if denom > 0 else np.nan
        ),
        "Mean_Bias_$": group["Error"].mean()
    })

print("\nOVERALL FORECAST COMPARISON")

overall = (
    matched.groupby("Model")
    .apply(evaluate, include_groups=False)
    .sort_values("WAPE_%")
)

display(overall.round(2))

print("\nWAPE BY STORE TYPE")

by_type = (
    matched.groupby(["Model", "Type"])
    .apply(evaluate, include_groups=False)
    .reset_index()
)

display(
    by_type.pivot(
        index="Model",
        columns="Type",
        values="WAPE_%"
    ).round(2)
)

# Save
overall.to_csv(
    base_path / "audit_forecast_model_comparison.csv"
)

by_type.to_csv(
    base_path / "audit_forecast_by_store_type.csv",
    index=False
)

print("\nForecast comparison saved.")


Common department-week observations: 42050
Models compared: 5

OVERALL FORECAST COMPARISON


,Observations,MAE_$,RMSE_$,WAPE_%,Mean_Bias_$
Model,,,,,
ExtraTrees,42050.0,1284.97,2637.97,7.83,178.23
HistGradientBoosting,42050.0,1352.28,2868.16,8.24,-82.32
MA4,42050.0,1628.51,3596.80,9.93,156.75
MA8,42050.0,1907.02,4134.08,11.63,313.88
MA12,42050.0,2069.60,4417.81,12.62,376.52



WAPE BY STORE TYPE


Type,A,B,C
Model,,,
ExtraTrees,7.43,9.23,5.44
HistGradientBoosting,7.83,9.57,6.31
MA12,11.95,15.28,6.97
MA4,9.44,11.73,6.55
MA8,11.02,14.01,6.73



Forecast comparison saved.


In [20]:

# CELL 17 — CHECK TRAINING DATA FOR 10-MODEL COMPARISON

import pandas as pd
import numpy as np
from IPython.display import display

raw_train = pd.read_csv(
    base_path / "train.csv",
    parse_dates=["Date"]
)

raw_features = pd.read_csv(
    base_path / "features.csv",
    parse_dates=["Date"]
)

raw_stores = pd.read_csv(
    base_path / "stores.csv"
)

cutoff = pd.Timestamp("2012-07-20")
test_end = pd.Timestamp("2012-10-26")

print("ORIGINAL WALMART DATA")
print("Sales rows:", len(raw_train))
print("Features rows:", len(raw_features))
print("Stores:", raw_stores["Store"].nunique())

print("\nAVAILABLE COLUMNS")
print("Sales:", raw_train.columns.tolist())
print("Features:", raw_features.columns.tolist())
print("Stores:", raw_stores.columns.tolist())

print("\nCHRONOLOGICAL SPLIT")
print("Training rows:",
      (raw_train["Date"] < cutoff).sum())

print("Evaluation rows:",
      raw_train["Date"].between(
          cutoff, test_end
      ).sum())

print("\nSTORE TYPE COVERAGE")

split_data = raw_train.merge(
    raw_stores[["Store", "Type"]],
    on="Store",
    validate="many_to_one"
)

split_data["Period"] = np.where(
    split_data["Date"] < cutoff,
    "Training",
    "Evaluation"
)

split_data = split_data[
    split_data["Date"] <= test_end
]

display(pd.crosstab(
    split_data["Type"],
    split_data["Period"]
))

print("\nMISSING VALUES IN FEATURES")
display(
    raw_features.isna().sum().sort_values(
        ascending=False
    )
)


ORIGINAL WALMART DATA
Sales rows: 421570
Features rows: 8190
Stores: 45

AVAILABLE COLUMNS
Sales: ['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday']
Features: ['Store', 'Date', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'IsHoliday']
Stores: ['Store', 'Type', 'Size']

CHRONOLOGICAL SPLIT
Training rows: 377155
Evaluation rows: 44415

STORE TYPE COVERAGE


Period,Evaluation,Training
Type,,
A,22645,192833
B,17154,146341
C,4616,37981



MISSING VALUES IN FEATURES


,0
MarkDown2,5269
MarkDown4,4726
MarkDown3,4577
MarkDown1,4158
MarkDown5,4140
CPI,585
Unemployment,585
Store,0
Fuel_Price,0
Temperature,0


Retraining an additional for double-check to match 10 models comparison

In [21]:

# CELL 18 — TRAIN GRADIENT BOOSTING REGRESSOR
# Walmart Final Model Evaluation

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

# 1. Reuse data loaded in Cell 17
data = raw_train.merge(
    raw_features.drop(columns=["IsHoliday"]),
    on=["Store", "Date"],
    how="left",
    validate="many_to_one"
)

data = data.merge(
    raw_stores,
    on="Store",
    how="left",
    validate="many_to_one"
)

data = data.sort_values(
    ["Store", "Dept", "Date"]
).reset_index(drop=True)

# 2. Build historical sales features
sales = data.groupby(["Store", "Dept"])["Weekly_Sales"]

data["Lag1"] = sales.shift(1)
data["Lag4"] = sales.shift(4)

data["MA4"] = sales.transform(
    lambda s: s.shift(1).rolling(4, min_periods=1).mean()
)

data["MA8"] = sales.transform(
    lambda s: s.shift(1).rolling(8, min_periods=1).mean()
)

data["MA12"] = sales.transform(
    lambda s: s.shift(1).rolling(12, min_periods=1).mean()
)

data["Month"] = data["Date"].dt.month
data["WeekOfYear"] = (
    data["Date"].dt.isocalendar().week.astype(int)
)

markdown_cols = [
    "MarkDown1", "MarkDown2", "MarkDown3",
    "MarkDown4", "MarkDown5"
]

data["Markdown_Recorded"] = (
    data[markdown_cols].notna().any(axis=1).astype(int)
)

data[markdown_cols] = data[markdown_cols].fillna(0)

# 3. Encode store type
data = pd.get_dummies(data, columns=["Type"], dtype=int)

features = [
    "Store", "Dept", "Size", "IsHoliday",
    "Temperature", "Fuel_Price", "CPI",
    "Unemployment", "Month", "WeekOfYear",
    "Lag1", "Lag4", "MA4", "MA8", "MA12",
    "Markdown_Recorded"
] + markdown_cols + ["Type_A", "Type_B", "Type_C"]

# 4. Chronological train/test
cutoff = pd.Timestamp("2012-07-20")
test_end = pd.Timestamp("2012-10-26")

train = data[data["Date"] < cutoff].copy()
test = data[data["Date"].between(cutoff, test_end)].copy()

X_train = train[features].astype(float)
y_train = train["Weekly_Sales"]

X_test = test[features].astype(float)
y_test = test["Weekly_Sales"]

# 5. Train only the new model
gb_model = make_pipeline(
    SimpleImputer(strategy="median"),
    GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
)

print("Training rows:", len(train))
print("Test rows:", len(test))
print("Training Gradient Boosting...")

gb_model.fit(X_train, y_train)

pred = gb_model.predict(X_test)

# 6. Evaluate
error = pred - y_test.to_numpy()

gb_results = pd.DataFrame([{
    "Model": "GradientBoosting",
    "Observations": len(test),
    "MAE": np.abs(error).mean(),
    "RMSE": np.sqrt(np.mean(error**2)),
    "WAPE_%": 100 * np.abs(error).sum() / np.abs(y_test).sum(),
    "Bias": error.mean()
}])

print("\nGRADIENT BOOSTING RESULTS")
display(gb_results.round(2))

# 7. Store-level type comparison
gb_predictions = test[
    ["Store", "Dept", "Date", "Weekly_Sales"]
].copy()

gb_predictions["Model"] = "GradientBoosting"
gb_predictions["Prediction"] = pred

gb_predictions = gb_predictions.merge(
    raw_stores[["Store", "Type"]],
    on="Store",
    validate="many_to_one"
)

def evaluate_type(g):
    err = g["Prediction"] - g["Weekly_Sales"]
    return pd.Series({
        "Observations": len(g),
        "MAE": err.abs().mean(),
        "WAPE_%": 100 * err.abs().sum() /
                  g["Weekly_Sales"].abs().sum()
    })

gb_by_type = (
    gb_predictions.groupby("Type")
    .apply(evaluate_type, include_groups=False)
    .reset_index()
)

print("\nRESULTS BY STORE TYPE")
display(gb_by_type.round(2))

# 8. Save to Google Drive
base_path = Path(BASE)

gb_results.to_csv(
    base_path / "gradient_boosting_final_results.csv",
    index=False
)

gb_predictions.to_csv(
    base_path / "gradient_boosting_predictions.csv",
    index=False
)

gb_by_type.to_csv(
    base_path / "gradient_boosting_store_type_results.csv",
    index=False
)

print("\nAll Gradient Boosting results saved.")


Training rows: 377155
Test rows: 44415
Training Gradient Boosting...

GRADIENT BOOSTING RESULTS


,Model,Observations,MAE,RMSE,WAPE_%,Bias
0,GradientBoosting,44415,1434.99,2991.15,9.21,82.66



RESULTS BY STORE TYPE


,Type,Observations,MAE,WAPE_%
0,A,22645.0,1703.52,8.67
1,B,17154.0,1269.90,10.70
2,C,4616.0,731.17,7.78



All Gradient Boosting results saved.


In [22]:

# CELL 19 — FINAL EXCEPTION PRIORITIZATION
# Walmart Markdown Performance Exception Investigation

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

base_path = Path(BASE)

# ------------------------------------------------
# 1. Reuse earlier evaluation results
# ------------------------------------------------

# 'comparison' was created in Cell 13:
# ET store-week results + IF-A and IF-B results

required = [
    "Store", "Date", "Type",
    "Expected_Sales", "Actual_Sales",
    "Positive_Net_Shortfall",
    "Flagged_Dept_Count",
    "Flagged_Dept_Rate",
    "ET_10pct", "ET_20pct",
    "IF_A_Flag", "IF_B_Flag"
]

if "comparison" not in globals():
    raise ValueError(
        "Please rerun Cell 13 to recreate 'comparison'."
    )

missing = [c for c in required if c not in comparison.columns]

if missing:
    print("Available columns:", comparison.columns.tolist())
    raise ValueError(f"Missing columns: {missing}")

priority = comparison.copy()
priority["Date"] = pd.to_datetime(priority["Date"])

assert not priority.duplicated(["Store", "Date"]).any()
assert len(priority) == 675, (
    "Expected 675 store-weeks in the evaluation period."
)

# ------------------------------------------------
# 2. Use ET + MAD to define eligible exceptions
# ------------------------------------------------

priority["Eligible"] = (
    priority["ET_10pct"].astype(bool)
    & (priority["Positive_Net_Shortfall"] > 0)
)

priority["IF_A_Flag"] = priority["IF_A_Flag"].astype(bool)
priority["IF_B_Flag"] = priority["IF_B_Flag"].astype(bool)

# IF is supporting evidence, not a mandatory filter
priority["IF_Agreement"] = np.select(
    [
        priority["IF_A_Flag"] & priority["IF_B_Flag"],
        priority["IF_A_Flag"] | priority["IF_B_Flag"]
    ],
    [
        "Both IF models",
        "One IF model"
    ],
    default="Neither IF model"
)

# ------------------------------------------------
# 3. Rank by modeled net sales shortfall
# ------------------------------------------------

eligible = priority[
    priority["Eligible"]
].copy()

eligible = eligible.sort_values(
    [
        "Positive_Net_Shortfall",
        "Flagged_Dept_Rate",
        "Store",
        "Date"
    ],
    ascending=[False, False, True, True]
).reset_index(drop=True)

eligible["Priority_Rank"] = (
    np.arange(1, len(eligible) + 1)
)

eligible["Review_Priority"] = np.where(
    eligible["Priority_Rank"] <= 20,
    "High — Top 20",
    "Backlog"
)

eligible["Estimated_Sales_Gap_$"] = (
    eligible["Positive_Net_Shortfall"]
)

eligible["Flagged_Dept_%"] = (
    eligible["Flagged_Dept_Rate"] * 100
)

# ------------------------------------------------
# 4. Final prioritized exception list
# ------------------------------------------------

columns = [
    "Priority_Rank",
    "Store",
    "Date",
    "Type",
    "Expected_Sales",
    "Actual_Sales",
    "Estimated_Sales_Gap_$",
    "Flagged_Dept_Count",
    "Flagged_Dept_%",
    "IF_A_Flag",
    "IF_B_Flag",
    "IF_Agreement",
    "Review_Priority"
]

top20 = eligible.head(20)[columns].copy()

print("TOP 20 WALMART EXCEPTIONS")
display(top20.round(2))

# ------------------------------------------------
# 5. Summary of review workload and gaps
# ------------------------------------------------

all_positive_gap = (
    priority["Positive_Net_Shortfall"].clip(lower=0).sum()
)

top20_gap = top20["Estimated_Sales_Gap_$"].sum()

summary = pd.DataFrame([{
    "Total_Store_Weeks": len(priority),
    "ET_10pct_Eligible": len(eligible),
    "Selected_For_Review": len(top20),
    "Total_Positive_Modeled_Gap_$": all_positive_gap,
    "Top20_Modeled_Gap_$": top20_gap,
    "Top20_Gap_Coverage_%": (
        100 * top20_gap / all_positive_gap
        if all_positive_gap > 0 else np.nan
    ),
    "Top20_IF_A_Agreement": int(
        top20["IF_A_Flag"].sum()
    ),
    "Top20_IF_B_Agreement": int(
        top20["IF_B_Flag"].sum()
    )
}])

print("\nREVIEW WORKLOAD SUMMARY")
display(summary.round(2))

print("\nTOP 20 BY STORE TYPE")
display(
    top20["Type"].value_counts()
    .reindex(["A", "B", "C"], fill_value=0)
    .rename_axis("Store_Type")
    .reset_index(name="Exceptions")
)

print("\nISOLATION FOREST AGREEMENT")
display(
    top20["IF_Agreement"]
    .value_counts()
    .rename_axis("Agreement")
    .reset_index(name="Store_Weeks")
)

# ------------------------------------------------
# 6. Select three illustrative cases
# ------------------------------------------------

# Case 1: Largest estimated net shortfall
case1 = eligible.iloc[[0]].copy()
case1["Case_Category"] = "Largest modeled gap"

# Case 2: Largest gap among cases supported by IF-A
candidates2 = eligible[
    eligible["IF_A_Flag"]
    & ~eligible.index.isin(case1.index)
]

case2 = candidates2.head(1).copy()
case2["Case_Category"] = "ET + IF agreement"

# Case 3: Largest gap not flagged by either IF model
used = pd.concat([case1, case2])[["Store", "Date"]]

candidates3 = eligible[
    ~eligible["IF_A_Flag"]
    & ~eligible["IF_B_Flag"]
].merge(
    used,
    on=["Store", "Date"],
    how="left",
    indicator=True
)

candidates3 = candidates3[
    candidates3["_merge"] == "left_only"
].drop(columns="_merge")

case3 = candidates3.head(1).copy()
case3["Case_Category"] = "ET-only disagreement"

three_cases = pd.concat(
    [case1, case2, case3],
    ignore_index=True
)

case_columns = [
    "Case_Category",
    "Store",
    "Date",
    "Type",
    "Expected_Sales",
    "Actual_Sales",
    "Positive_Net_Shortfall",
    "Flagged_Dept_Rate",
    "IF_A_Flag",
    "IF_B_Flag"
]

print("\nTHREE CASES FOR FINAL PRESENTATION")
display(three_cases[case_columns].round(2))

# ------------------------------------------------
# 7. Save CSV and Excel outputs
# ------------------------------------------------

eligible.to_csv(
    base_path / "final_prioritized_exceptions.csv",
    index=False
)

top20.to_csv(
    base_path / "final_top20_exceptions.csv",
    index=False
)

three_cases.to_csv(
    base_path / "final_three_exception_cases.csv",
    index=False
)

output_file = base_path / "Walmart_Final_Exception_Prioritization.xlsx"

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    summary.to_excel(
        writer, sheet_name="Summary", index=False
    )

    top20.to_excel(
        writer, sheet_name="Top20", index=False
    )

    three_cases.to_excel(
        writer, sheet_name="Three_Cases", index=False
    )

    eligible.to_excel(
        writer, sheet_name="All_Eligible", index=False
    )

print("\nSaved:", output_file)
print("Final exception prioritization completed.")


TOP 20 WALMART EXCEPTIONS


,Priority_Rank,Store,Date,Type,Expected_Sales,Actual_Sales,Estimated_Sales_Gap_$,Flagged_Dept_Count,Flagged_Dept_%,IF_A_Flag,IF_B_Flag,IF_Agreement,Review_Priority
0,1,27,2012-09-14,A,1751935.29,1511914.98,240020.31,13,18.84,False,False,Neither IF model,High — Top 20
1,2,13,2012-09-14,A,2106252.21,1918420.24,187831.97,13,18.57,False,False,Neither IF model,High — Top 20
2,3,23,2012-09-07,B,1597271.28,1426508.44,170762.84,14,20.90,False,False,Neither IF model,High — Top 20
3,4,6,2012-09-14,A,1537680.99,1372402.58,165278.41,14,20.00,False,False,Neither IF model,High — Top 20
4,5,27,2012-07-27,A,1789837.09,1625040.49,164796.60,9,13.24,False,False,Neither IF model,High — Top 20
5,6,14,2012-09-14,A,1713396.21,1553717.90,159678.31,12,17.65,False,False,Neither IF model,High — Top 20
6,7,10,2012-08-31,B,1806634.20,1649822.20,156812.00,7,10.29,True,False,One IF model,High — Top 20
7,8,39,2012-09-14,A,1603181.82,1447604.08,155577.74,12,17.65,False,False,Neither IF model,High — Top 20
8,9,23,2012-09-14,B,1421143.15,1277761.47,143381.68,7,10.45,False,True,One IF model,High — Top 20
9,10,13,2012-07-27,A,2056709.70,1913813.59,142896.11,7,10.00,False,False,Neither IF model,High — Top 20



REVIEW WORKLOAD SUMMARY


,Total_Store_Weeks,ET_10pct_Eligible,Selected_For_Review,Total_Positive_Modeled_Gap_$,Top20_Modeled_Gap_$,Top20_Gap_Coverage_%,Top20_IF_A_Agreement,Top20_IF_B_Agreement
0,675,251,20,15310079.54,2913592.41,19.03,4,3



TOP 20 BY STORE TYPE


,Store_Type,Exceptions
0,A,15
1,B,5
2,C,0



ISOLATION FOREST AGREEMENT


,Agreement,Store_Weeks
0,Neither IF model,15
1,One IF model,3
2,Both IF models,2



THREE CASES FOR FINAL PRESENTATION


,Case_Category,Store,Date,Type,Expected_Sales,Actual_Sales,Positive_Net_Shortfall,Flagged_Dept_Rate,IF_A_Flag,IF_B_Flag
0,Largest modeled gap,27,2012-09-14,A,1751935.29,1511914.98,240020.31,0.19,False,False
1,ET + IF agreement,10,2012-08-31,B,1806634.20,1649822.20,156812.00,0.10,True,False
2,ET-only disagreement,13,2012-09-14,A,2106252.21,1918420.24,187831.97,0.19,False,False



Saved: /content/drive/MyDrive/Walmart Recruiting Sales Data/Walmart_Final_Exception_Prioritization.xlsx
Final exception prioritization completed.


In [23]:

# CELL 20 — SELECT THREE DISTINCT EXCEPTION CASES

import pandas as pd
import numpy as np
from IPython.display import display

# Use completed Cell 19 results
cases = eligible.copy()

cases["IF_A_Flag"] = cases["IF_A_Flag"].astype(bool)
cases["IF_B_Flag"] = cases["IF_B_Flag"].astype(bool)

# Three distinct agreement patterns
case_groups = {
    "ET only": cases[
        ~cases["IF_A_Flag"] &
        ~cases["IF_B_Flag"]
    ],
    "ET + one IF": cases[
        cases["IF_A_Flag"] ^
        cases["IF_B_Flag"]
    ],
    "ET + both IF": cases[
        cases["IF_A_Flag"] &
        cases["IF_B_Flag"]
    ]
}

selected = []

for label, group in case_groups.items():
    if group.empty:
        print(f"No case available for: {label}")
        continue

    chosen = group.sort_values(
        "Positive_Net_Shortfall",
        ascending=False
    ).head(1).copy()

    chosen["Case_Category"] = label
    selected.append(chosen)

three_cases_final = pd.concat(
    selected,
    ignore_index=True
)

display_cols = [
    "Case_Category",
    "Store",
    "Date",
    "Type",
    "Expected_Sales",
    "Actual_Sales",
    "Positive_Net_Shortfall",
    "Flagged_Dept_Rate",
    "IF_A_Flag",
    "IF_B_Flag"
]

print("FINAL THREE EXCEPTION CASES")
display(three_cases_final[display_cols].round(2))

# Export
three_cases_final.to_csv(
    base_path / "final_three_cases_distinct.csv",
    index=False
)

print("\nThree-case selection saved.")


FINAL THREE EXCEPTION CASES


,Case_Category,Store,Date,Type,Expected_Sales,Actual_Sales,Positive_Net_Shortfall,Flagged_Dept_Rate,IF_A_Flag,IF_B_Flag
0,ET only,27,2012-09-14,A,1751935.29,1511914.98,240020.31,0.19,False,False
1,ET + one IF,10,2012-08-31,B,1806634.20,1649822.20,156812.00,0.10,True,False
2,ET + both IF,28,2012-09-14,A,1241459.64,1124561.04,116898.60,0.16,True,True



Three-case selection saved.


In [24]:

# CELL 21 — INVESTIGATE THREE WALMART EXCEPTIONS
# Department-level evidence, markdowns, and sales history

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

base_path = Path(BASE)

# ------------------------------------------------
# 1. Load selected cases and department results
# ------------------------------------------------

cases = three_cases_final[
    ["Case_Category", "Store", "Date", "Type"]
].copy()

cases["Date"] = pd.to_datetime(cases["Date"])

dept_data = pd.read_csv(
    base_path / "et_mad_department_week.csv",
    parse_dates=["Date"]
)

# Check columns before analysis
print("Department data columns:")
print(dept_data.columns.tolist())

required = [
    "Store", "Dept", "Date",
    "Expected_Sales", "Actual_Sales"
]

missing = [c for c in required if c not in dept_data.columns]

if missing:
    raise ValueError(f"Missing department columns: {missing}")

# ------------------------------------------------
# 2. Find department-level contributors
# ------------------------------------------------

case_depts = dept_data.merge(
    cases,
    on=["Store", "Date"],
    how="inner",
    suffixes=("", "_Case")
)

case_depts["Sales_Gap"] = (
    case_depts["Expected_Sales"] -
    case_depts["Actual_Sales"]
)

case_depts["Positive_Dept_Gap"] = (
    case_depts["Sales_Gap"].clip(lower=0)
)

case_depts["Gap_Percent"] = np.where(
    case_depts["Expected_Sales"] > 0,
    100 * case_depts["Sales_Gap"]
    / case_depts["Expected_Sales"],
    np.nan
)

top_departments = (
    case_depts.sort_values(
        ["Store", "Date", "Positive_Dept_Gap"],
        ascending=[True, True, False]
    )
    .groupby(["Store", "Date"])
    .head(5)
    .copy()
)

print("\nTOP FIVE DEPARTMENT CONTRIBUTORS PER CASE")

display(
    top_departments[
        [
            "Store", "Date", "Dept",
            "Expected_Sales", "Actual_Sales",
            "Sales_Gap", "Positive_Dept_Gap",
            "Gap_Percent"
        ]
    ].round(2)
)

# ------------------------------------------------
# 3. Retrieve markdown and economic context
# ------------------------------------------------

features = raw_features.copy()
features["Date"] = pd.to_datetime(features["Date"])

markdown_cols = [
    "MarkDown1", "MarkDown2", "MarkDown3",
    "MarkDown4", "MarkDown5"
]

context_cols = [
    "Store", "Date", "IsHoliday",
    "Temperature", "Fuel_Price",
    "CPI", "Unemployment"
] + markdown_cols

context = cases.merge(
    features[context_cols],
    on=["Store", "Date"],
    how="left",
    validate="one_to_one"
)

context["Markdown_Fields_Recorded"] = (
    context[markdown_cols].notna().sum(axis=1)
)

context["Total_Recorded_Markdown"] = (
    context[markdown_cols].sum(axis=1, min_count=1)
)

context["Markdown_Data_Available"] = (
    context["Markdown_Fields_Recorded"] > 0
)

print("\nMARKDOWN AND ECONOMIC CONTEXT")

display(
    context[
        [
            "Case_Category", "Store", "Date",
            "Type", "IsHoliday",
            "Markdown_Fields_Recorded",
            "Total_Recorded_Markdown",
            "Temperature", "Fuel_Price",
            "CPI", "Unemployment"
        ]
    ].round(2)
)

# ------------------------------------------------
# 4. Historical store sales comparison
# ------------------------------------------------

sales_history = raw_train.groupby(
    ["Store", "Date"],
    as_index=False
)["Weekly_Sales"].sum()

sales_history = sales_history.sort_values(
    ["Store", "Date"]
)

history_rows = []

for _, case in cases.iterrows():

    store = case["Store"]
    date = case["Date"]

    historical = sales_history[
        (sales_history["Store"] == store)
        & (sales_history["Date"] < date)
    ].tail(8)

    current = sales_history[
        (sales_history["Store"] == store)
        & (sales_history["Date"] == date)
    ]

    if historical.empty or current.empty:
        continue

    baseline = historical["Weekly_Sales"].mean()
    actual = current["Weekly_Sales"].iloc[0]

    history_rows.append({
        "Case_Category": case["Case_Category"],
        "Store": store,
        "Date": date,
        "Prior_Weeks": len(historical),
        "Prior_8_Week_Avg": baseline,
        "Actual_Store_Sales": actual,
        "Deviation_From_8W_%": (
            100 * (actual - baseline) / baseline
            if baseline != 0 else np.nan
        )
    })

history_summary = pd.DataFrame(history_rows)

print("\nHISTORICAL STORE SALES COMPARISON")
display(history_summary.round(2))

# ------------------------------------------------
# 5. Combine case-level investigation evidence
# ------------------------------------------------

case_summary = three_cases_final.copy()

case_summary = case_summary.merge(
    context.drop(columns=["Type", "Case_Category"]),
    on=["Store", "Date"],
    how="left",
    validate="one_to_one"
)

case_summary = case_summary.merge(
    history_summary[
        [
            "Store", "Date",
            "Prior_8_Week_Avg",
            "Deviation_From_8W_%"
        ]
    ],
    on=["Store", "Date"],
    how="left",
    validate="one_to_one"
)

case_summary["Investigation_Status"] = (
    "Requires analyst review"
)

case_summary["Recommended_Next_Step"] = (
    "Review department drivers, recorded markdowns, "
    "historical sales, and promotional context."
)

print("\nFINAL THREE-CASE INVESTIGATION SUMMARY")

display(
    case_summary[
        [
            "Case_Category", "Store", "Date",
            "Type", "Positive_Net_Shortfall",
            "Flagged_Dept_Rate",
            "Total_Recorded_Markdown",
            "Deviation_From_8W_%",
            "Investigation_Status"
        ]
    ].round(2)
)

# ------------------------------------------------
# 6. Export investigation evidence
# ------------------------------------------------

output_file = (
    base_path / "Walmart_Three_Case_Investigations.xlsx"
)

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    case_summary.to_excel(
        writer,
        sheet_name="Case_Summary",
        index=False
    )

    top_departments.to_excel(
        writer,
        sheet_name="Top_Departments",
        index=False
    )

    case_depts.to_excel(
        writer,
        sheet_name="All_Case_Departments",
        index=False
    )

    context.to_excel(
        writer,
        sheet_name="Markdown_Context",
        index=False
    )

    history_summary.to_excel(
        writer,
        sheet_name="Sales_History",
        index=False
    )

print("\nInvestigation saved:")
print(output_file)


Department data columns:
['Store', 'Dept', 'Date', 'Type', 'Actual_Sales', 'Expected_Sales', 'Dollar_Shortfall', 'Shortfall_Pct', 'Flag_GlobalMAD', 'Flag_TypeMAD', 'Flag_Fixed10', 'Flag_P90', 'Flag_P95']

TOP FIVE DEPARTMENT CONTRIBUTORS PER CASE


,Store,Date,Dept,Expected_Sales,Actual_Sales,Sales_Gap,Positive_Dept_Gap,Gap_Percent
2,10,2012-08-31,3,44712.53,27278.21,17434.32,17434.32,38.99
52,10,2012-08-31,72,112432.47,96347.84,16084.63,16084.63,14.31
1,10,2012-08-31,2,109963.31,95671.50,14291.81,14291.81,13.00
54,10,2012-08-31,79,42616.49,31057.77,11558.72,11558.72,27.12
20,10,2012-08-31,23,64909.55,54524.55,10385.00,10385.00,16.00
70,27,2012-09-14,3,59063.46,24075.61,34987.85,34987.85,59.24
132,27,2012-09-14,95,125002.89,107557.38,17445.51,17445.51,13.96
118,27,2012-09-14,72,54241.65,38844.83,15396.82,15396.82,28.39
69,27,2012-09-14,2,83775.80,71725.18,12050.62,12050.62,14.38
76,27,2012-09-14,9,33277.39,23225.46,10051.93,10051.93,30.21



MARKDOWN AND ECONOMIC CONTEXT


,Case_Category,Store,Date,Type,IsHoliday,Markdown_Fields_Recorded,Total_Recorded_Markdown,Temperature,Fuel_Price,CPI,Unemployment
0,ET only,27,2012-09-14,A,False,5,23812.24,68.72,4.09,142.59,8.24
1,ET + one IF,10,2012-08-31,B,False,4,42272.28,86.97,3.89,130.89,7.17
2,ET + both IF,28,2012-09-14,A,False,5,28371.49,83.64,4.13,130.98,10.93



HISTORICAL STORE SALES COMPARISON


,Case_Category,Store,Date,Prior_Weeks,Prior_8_Week_Avg,Actual_Store_Sales,Deviation_From_8W_%
0,ET only,27,2012-09-14,8,1733977.39,1519604.50,-12.36
1,ET + one IF,10,2012-08-31,8,1833975.12,1650285.54,-10.02
2,ET + both IF,28,2012-09-14,8,1251302.10,1124660.77,-10.12



FINAL THREE-CASE INVESTIGATION SUMMARY


,Case_Category,Store,Date,Type,Positive_Net_Shortfall,Flagged_Dept_Rate,Total_Recorded_Markdown,Deviation_From_8W_%,Investigation_Status
0,ET only,27,2012-09-14,A,240020.31,0.19,23812.24,-12.36,Requires analyst review
1,ET + one IF,10,2012-08-31,B,156812.00,0.10,42272.28,-10.02,Requires analyst review
2,ET + both IF,28,2012-09-14,A,116898.60,0.16,28371.49,-10.12,Requires analyst review



Investigation saved:
/content/drive/MyDrive/Walmart Recruiting Sales Data/Walmart_Three_Case_Investigations.xlsx


In [25]:

# CELL 22 — FINAL BUSINESS DECISION & AI AGENT EVIDENCE
# Walmart Markdown Performance Exception Investigation

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

base_path = Path(BASE)

# ------------------------------------------------
# 1. Reuse investigation outputs from Cell 21
# ------------------------------------------------

required_vars = [
    "case_summary",
    "top_departments",
    "context",
    "history_summary"
]

missing_vars = [
    name for name in required_vars
    if name not in globals()
]

if missing_vars:
    raise ValueError(
        f"Run Cell 21 first. Missing: {missing_vars}"
    )

agent_cases = case_summary.copy()

# ------------------------------------------------
# 2. Create model agreement indicators
# ------------------------------------------------

agent_cases["IF_A_Flag"] = (
    agent_cases["IF_A_Flag"].astype(bool)
)

agent_cases["IF_B_Flag"] = (
    agent_cases["IF_B_Flag"].astype(bool)
)

agent_cases["IF_Models_Agree"] = (
    agent_cases["IF_A_Flag"].astype(int)
    + agent_cases["IF_B_Flag"].astype(int)
)

agent_cases["Model_Agreement"] = (
    agent_cases["IF_Models_Agree"].map({
        0: "ET only",
        1: "ET + one IF",
        2: "ET + both IF"
    })
)

# ------------------------------------------------
# 3. Determine analyst review priorities
# ------------------------------------------------

# Relative to the three selected cases only
agent_cases["Gap_Rank"] = (
    agent_cases["Positive_Net_Shortfall"]
    .rank(method="first", ascending=False)
    .astype(int)
)

def assign_priority(row):

    if row["Gap_Rank"] == 1:
        return "High — Largest modeled gap"

    if row["IF_Models_Agree"] == 2:
        return "High — Multiple model signals"

    return "Standard — Investigate"

agent_cases["Review_Priority"] = (
    agent_cases.apply(assign_priority, axis=1)
)

agent_cases["Decision"] = "Investigate"

# ------------------------------------------------
# 4. Summarize top department contributors
# ------------------------------------------------

top3 = (
    top_departments.sort_values(
        ["Store", "Date", "Positive_Dept_Gap"],
        ascending=[True, True, False]
    )
    .groupby(["Store", "Date"])
    .head(3)
    .copy()
)

top3["Dept_Evidence"] = top3.apply(
    lambda r: (
        f"Dept {int(r['Dept'])}: "
        f"${r['Positive_Dept_Gap']:,.0f} "
        f"modeled gap"
    ),
    axis=1
)

department_evidence = (
    top3.groupby(["Store", "Date"])["Dept_Evidence"]
    .agg("; ".join)
    .reset_index()
)

agent_cases = agent_cases.merge(
    department_evidence,
    on=["Store", "Date"],
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------
# 5. Create AI agent evidence explanations
# ------------------------------------------------

def build_explanation(row):

    markdown_text = (
        f"${row['Total_Recorded_Markdown']:,.0f}"
        if pd.notna(row["Total_Recorded_Markdown"])
        else "unavailable"
    )

    historical_text = (
        f"{row['Deviation_From_8W_%']:.1f}%"
        if pd.notna(row["Deviation_From_8W_%"])
        else "unavailable"
    )

    return (
        f"Store {int(row['Store'])} on "
        f"{row['Date']:%Y-%m-%d} was flagged "
        f"by Extra Trees + MAD. "
        f"Estimated net sales gap: "
        f"${row['Positive_Net_Shortfall']:,.0f}. "
        f"Flagged department rate: "
        f"{row['Flagged_Dept_Rate']:.1%}. "
        f"Model agreement: {row['Model_Agreement']}. "
        f"Recorded markdown total: {markdown_text}. "
        f"Sales deviation from prior 8-week average: "
        f"{historical_text}. "
        f"Leading department contributors: "
        f"{row['Dept_Evidence']}."
    )

agent_cases["Evidence_Summary"] = (
    agent_cases.apply(build_explanation, axis=1)
)

# ------------------------------------------------
# 6. Recommend next analyst actions
# ------------------------------------------------

def recommend_action(row):

    if row["IF_Models_Agree"] == 2:
        return (
            "Review department-level drivers and "
            "validate the anomaly against operational "
            "records. Multiple detection signals agree, "
            "but markdown causality is unverified."
        )

    if row["Gap_Rank"] == 1:
        return (
            "Prioritize review of the large modeled "
            "sales gap. Check markdown execution, "
            "inventory availability, pricing, and "
            "department-level sales patterns."
        )

    return (
        "Review leading underperforming departments, "
        "historical sales, and recorded markdowns. "
        "Determine whether operational follow-up "
        "is warranted."
    )

agent_cases["Recommended_Action"] = (
    agent_cases.apply(recommend_action, axis=1)
)

agent_cases["Causality_Status"] = (
    "Markdown impact not established"
)

agent_cases["Human_Review_Required"] = True

# ------------------------------------------------
# 7. Create final analyst-facing table
# ------------------------------------------------

final_columns = [
    "Case_Category",
    "Store",
    "Date",
    "Type",
    "Positive_Net_Shortfall",
    "Flagged_Dept_Rate",
    "Model_Agreement",
    "Total_Recorded_Markdown",
    "Deviation_From_8W_%",
    "Review_Priority",
    "Decision",
    "Dept_Evidence",
    "Evidence_Summary",
    "Recommended_Action",
    "Causality_Status",
    "Human_Review_Required"
]

agent_report = agent_cases[final_columns].copy()

print("FINAL WALMART AI AGENT DECISION REPORT")

display(
    agent_report[
        [
            "Store", "Date",
            "Positive_Net_Shortfall",
            "Model_Agreement",
            "Review_Priority",
            "Decision"
        ]
    ].round(2)
)

print("\nEXPLANATIONS AND ACTIONS")

for _, row in agent_report.iterrows():

    print("\n" + "=" * 65)
    print(f"STORE {int(row['Store'])}")
    print("=" * 65)
    print("Priority:", row["Review_Priority"])
    print("Decision:", row["Decision"])
    print("\nEvidence:")
    print(row["Evidence_Summary"])
    print("\nRecommended action:")
    print(row["Recommended_Action"])

# ------------------------------------------------
# 8. Save agent-ready report
# ------------------------------------------------

output_file = (
    base_path / "Walmart_Final_AI_Agent_Evidence.xlsx"
)

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    agent_report.to_excel(
        writer,
        sheet_name="Agent_Decisions",
        index=False
    )

    case_summary.to_excel(
        writer,
        sheet_name="Case_Evidence",
        index=False
    )

    top_departments.to_excel(
        writer,
        sheet_name="Department_Drivers",
        index=False
    )

    context.to_excel(
        writer,
        sheet_name="Markdown_Context",
        index=False
    )

    history_summary.to_excel(
        writer,
        sheet_name="Historical_Sales",
        index=False
    )

agent_report.to_csv(
    base_path / "walmart_agent_decisions.csv",
    index=False
)

print("\nSaved:")
print(output_file)
print("Walmart AI agent evidence report completed.")


FINAL WALMART AI AGENT DECISION REPORT


,Store,Date,Positive_Net_Shortfall,Model_Agreement,Review_Priority,Decision
0,27,2012-09-14,240020.31,ET only,High — Largest modeled gap,Investigate
1,10,2012-08-31,156812.00,ET + one IF,Standard — Investigate,Investigate
2,28,2012-09-14,116898.60,ET + both IF,High — Multiple model signals,Investigate



EXPLANATIONS AND ACTIONS

STORE 27
Priority: High — Largest modeled gap
Decision: Investigate

Evidence:
Store 27 on 2012-09-14 was flagged by Extra Trees + MAD. Estimated net sales gap: $240,020. Flagged department rate: 18.8%. Model agreement: ET only. Recorded markdown total: $23,812. Sales deviation from prior 8-week average: -12.4%. Leading department contributors: Dept 3: $34,988 modeled gap; Dept 95: $17,446 modeled gap; Dept 72: $15,397 modeled gap.

Recommended action:
Prioritize review of the large modeled sales gap. Check markdown execution, inventory availability, pricing, and department-level sales patterns.

STORE 10
Priority: Standard — Investigate
Decision: Investigate

Evidence:
Store 10 on 2012-08-31 was flagged by Extra Trees + MAD. Estimated net sales gap: $156,812. Flagged department rate: 10.3%. Model agreement: ET + one IF. Recorded markdown total: $42,272. Sales deviation from prior 8-week average: -10.0%. Leading department contributors: Dept 3: $17,434 modele

In [26]:

# CELL 23A — CHECK HISTORICAL ET PREDICTIONS

import pandas as pd
from pathlib import Path

base_path = Path(BASE)
cutoff = pd.Timestamp("2012-07-20")

print("CHECKING AVAILABLE MODEL FILES\n")

for path in sorted(base_path.glob("*.csv")):

    try:
        columns = pd.read_csv(path, nrows=0).columns.tolist()

        date_col = next(
            (c for c in ["Date", "date", "Week"]
             if c in columns),
            None
        )

        prediction_cols = [
            c for c in columns
            if any(word in c.lower()
                   for word in ["predict", "expected", "residual"])
        ]

        if date_col and prediction_cols:

            dates = pd.read_csv(
                path,
                usecols=[date_col]
            )

            dates[date_col] = pd.to_datetime(
                dates[date_col], errors="coerce"
            )

            print(f"FILE: {path.name}")
            print("Prediction columns:", prediction_cols)
            print("Earliest date:", dates[date_col].min())
            print("Latest date:", dates[date_col].max())
            print(
                "Rows before evaluation:",
                (dates[date_col] < cutoff).sum()
            )
            print("-" * 55)

    except Exception as error:
        print(f"Could not inspect {path.name}: {error}")

print("\nCheck completed.")


CHECKING AVAILABLE MODEL FILES

FILE: FINAL_hybrid_top20_cases.csv
Prediction columns: ['ET_Expected_Eligible_Sales']
Earliest date: 2012-07-27 00:00:00
Latest date: 2012-10-12 00:00:00
Rows before evaluation: 0
-------------------------------------------------------
FILE: additional_benchmarks_predictions.csv
Prediction columns: ['Prediction']
Earliest date: 2012-07-20 00:00:00
Latest date: 2012-10-26 00:00:00
Rows before evaluation: 0
-------------------------------------------------------
FILE: audit_equal_budget_top20.csv
Prediction columns: ['Expected_Sales', 'MA4_Residual_Pct']
Earliest date: 2012-07-20 00:00:00
Latest date: 2012-10-26 00:00:00
Rows before evaluation: 0
-------------------------------------------------------
FILE: audit_et_if_business_comparison.csv
Prediction columns: ['Expected_Sales', 'MA4_Residual_Pct']
Earliest date: 2012-07-20 00:00:00
Latest date: 2012-10-26 00:00:00
Rows before evaluation: 0
-------------------------------------------------------
FILE: au

In [27]:

# CELL 23B — CHRONOLOGICAL EXTRA TREES RESIDUALS

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from IPython.display import display

BASE = Path("/content/drive/MyDrive/Walmart Recruiting Sales Data")

# ------------------------------------------
# 1. Rebuild original ET feature engineering
# ------------------------------------------

sales = pd.read_csv(BASE / "train.csv", parse_dates=["Date"])
factors = pd.read_csv(BASE / "features.csv", parse_dates=["Date"])
stores = pd.read_csv(BASE / "stores.csv")

data = (
    sales.merge(
        factors,
        on=["Store", "Date", "IsHoliday"],
        how="left",
        validate="many_to_one"
    )
    .merge(stores, on="Store", validate="many_to_one")
    .sort_values(["Store", "Dept", "Date"])
    .reset_index(drop=True)
)

data["Year"] = data["Date"].dt.year
data["Month"] = data["Date"].dt.month
data["Week"] = data["Date"].dt.isocalendar().week.astype(int)
data["Week_Sin"] = np.sin(2 * np.pi * data["Week"] / 52)
data["Week_Cos"] = np.cos(2 * np.pi * data["Week"] / 52)
data["IsHoliday"] = data["IsHoliday"].astype(int)

for t in ["A", "B", "C"]:
    data[f"Type_{t}"] = (data["Type"] == t).astype(int)

# Construct calendar-aligned historical features
data["Observed_Row"] = True
parts = []

for (store, dept), group in data.groupby(["Store", "Dept"], sort=False):

    weekly = (
        group.set_index("Date")
        .sort_index()
        .asfreq("W-FRI")
    )

    weekly["Store"] = store
    weekly["Dept"] = dept

    history = pd.to_numeric(
        weekly["Weekly_Sales"], errors="coerce"
    )

    for lag in [1, 2, 4, 8]:
        weekly[f"Lag_{lag}"] = history.shift(lag)

    for window in [4, 8]:
        weekly[f"MA_{window}"] = (
            history.shift(1)
            .rolling(window, min_periods=window)
            .mean()
        )

    parts.append(weekly.reset_index())

et_data = pd.concat(parts, ignore_index=True)

required_lags = [
    "Lag_1", "Lag_2", "Lag_4", "Lag_8",
    "MA_4", "MA_8"
]

et_data = (
    et_data[et_data["Observed_Row"].eq(True)]
    .dropna(subset=required_lags)
    .copy()
)

feature_cols = [
    "Store", "Dept", "Size", "IsHoliday",
    "Year", "Month", "Week",
    "Week_Sin", "Week_Cos",
    "Type_A", "Type_B", "Type_C",
    "Temperature", "Fuel_Price",
    "CPI", "Unemployment",
    "MarkDown1", "MarkDown2", "MarkDown3",
    "MarkDown4", "MarkDown5",
    "Lag_1", "Lag_2", "Lag_4", "Lag_8",
    "MA_4", "MA_8"
]

assert not et_data.duplicated(
    ["Store", "Dept", "Date"]
).any()

print("Eligible observations:", len(et_data))

# ------------------------------------------
# 2. Original Extra Trees model settings
# ------------------------------------------

def new_et_model():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        ExtraTreesRegressor(
            n_estimators=150,
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=42
        )
    )

# ------------------------------------------
# 3. Chronological out-of-sample validation
# ------------------------------------------

validation_windows = [
    ("Validation 1", "2011-12-02", "2012-02-10"),
    ("Validation 2", "2012-02-17", "2012-04-27"),
    ("Validation 3", "2012-05-04", "2012-07-13")
]

validation_results = []

for label, start, end in validation_windows:

    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    train_fold = et_data[et_data["Date"] < start]
    valid_fold = et_data[
        et_data["Date"].between(start, end)
    ].copy()

    assert len(train_fold) > 0
    assert len(valid_fold) > 0
    assert train_fold["Date"].max() < valid_fold["Date"].min()

    print(f"\n{label}")
    print("Training:", len(train_fold))
    print("Validation:", len(valid_fold))

    model = new_et_model()

    model.fit(
        train_fold[feature_cols],
        train_fold["Weekly_Sales"]
    )

    valid_fold["Prediction"] = model.predict(
        valid_fold[feature_cols]
    )

    valid_fold["Residual"] = (
        valid_fold["Weekly_Sales"]
        - valid_fold["Prediction"]
    )

    valid_fold["Validation_Window"] = label

    validation_results.append(
        valid_fold[
            [
                "Store", "Dept", "Date", "Type",
                "Weekly_Sales", "Prediction",
                "Residual", "Validation_Window"
            ]
        ]
    )

historical_et = pd.concat(
    validation_results, ignore_index=True
)

assert historical_et["Date"].max() < pd.Timestamp("2012-07-20")

# ------------------------------------------
# 4. Train on all pre-test history
# ------------------------------------------

cutoff = pd.Timestamp("2012-07-20")
test_end = pd.Timestamp("2012-10-26")

final_train = et_data[et_data["Date"] < cutoff]

final_test = et_data[
    et_data["Date"].between(cutoff, test_end)
].copy()

print("\nFINAL HOLDOUT")
print("Training:", len(final_train))
print("Test:", len(final_test))

final_model = new_et_model()

final_model.fit(
    final_train[feature_cols],
    final_train["Weekly_Sales"]
)

final_test["Prediction"] = final_model.predict(
    final_test[feature_cols]
)

final_test["Residual"] = (
    final_test["Weekly_Sales"]
    - final_test["Prediction"]
)

final_et = final_test[
    [
        "Store", "Dept", "Date", "Type",
        "Weekly_Sales", "Prediction", "Residual"
    ]
].copy()

# ------------------------------------------
# 5. Verify chronological separation
# ------------------------------------------

print("\nHISTORICAL ET RESIDUALS")
print("Rows:", len(historical_et))
print("Date range:",
      historical_et["Date"].min(),
      "to",
      historical_et["Date"].max())

print("\nFINAL ET TEST")
print("Rows:", len(final_et))
print("Date range:",
      final_et["Date"].min(),
      "to",
      final_et["Date"].max())

assert historical_et["Date"].max() < final_et["Date"].min()

# ------------------------------------------
# 6. Save results
# ------------------------------------------

historical_et.to_csv(
    BASE / "historical_et_oos_residuals.csv",
    index=False
)

final_et.to_csv(
    BASE / "final_et_holdout_for_mad.csv",
    index=False
)

print("\nHistorical and final ET predictions saved.")


Eligible observations: 382364

Validation 1
Training: 245719
Validation: 31550

Validation 2
Training: 277269
Validation: 31493

Validation 3
Training: 308762
Validation: 31200

FINAL HOLDOUT
Training: 339962
Test: 42402

HISTORICAL ET RESIDUALS
Rows: 94243
Date range: 2011-12-02 00:00:00 to 2012-07-13 00:00:00

FINAL ET TEST
Rows: 42402
Date range: 2012-07-20 00:00:00 to 2012-10-26 00:00:00

Historical and final ET predictions saved.


In [28]:

# CELL 23C — GLOBAL VS TYPE-SPECIFIC MAD EVALUATION
# Walmart Markdown Exception Investigation

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

BASE = Path("/content/drive/MyDrive/Walmart Recruiting Sales Data")

# --------------------------------------------------
# 1. Load historical and final-test ET predictions
# --------------------------------------------------

hist = pd.read_csv(
    BASE / "historical_et_oos_residuals.csv",
    parse_dates=["Date"]
)

test = pd.read_csv(
    BASE / "final_et_holdout_for_mad.csv",
    parse_dates=["Date"]
)

assert hist["Date"].max() < test["Date"].min()

assert not test.duplicated(
    ["Store", "Dept", "Date"]
).any()

print("Historical calibration rows:", len(hist))
print("Final test rows:", len(test))

# --------------------------------------------------
# 2. Calculate MAD thresholds using HISTORY ONLY
# --------------------------------------------------

MAD_MULTIPLIER = 3.0
MAD_SCALE = 1.4826

def calculate_mad(x):

    x = pd.to_numeric(x, errors="coerce").dropna()

    median = x.median()
    mad = np.median(np.abs(x - median))

    threshold = (
        median - MAD_MULTIPLIER * MAD_SCALE * mad
    )

    return pd.Series({
        "Historical_N": len(x),
        "Median_Residual": median,
        "MAD": mad,
        "Negative_Threshold": threshold
    })

# One threshold for all store types
global_stats = calculate_mad(hist["Residual"])

global_threshold = global_stats["Negative_Threshold"]

# Separate threshold for A, B, C
type_stats = (
    hist.groupby("Type")["Residual"]
    .apply(calculate_mad)
    .unstack()
)

print("\nGLOBAL MAD THRESHOLD")
display(global_stats.to_frame("Global").T.round(2))

print("\nTYPE-SPECIFIC MAD THRESHOLDS")
display(type_stats.round(2))

# --------------------------------------------------
# 3. Apply both thresholds to the final test
# --------------------------------------------------

test["Global_MAD_Flag"] = (
    test["Residual"] < global_threshold
)

test["Type_Threshold"] = test["Type"].map(
    type_stats["Negative_Threshold"]
)

test["Type_MAD_Flag"] = (
    test["Residual"] < test["Type_Threshold"]
)

test["Positive_Gap"] = (
    test["Prediction"] - test["Weekly_Sales"]
).clip(lower=0)

# --------------------------------------------------
# 4. Compare department-level exception rates
# --------------------------------------------------

department_results = []

for store_type, group in test.groupby("Type"):

    for method, flag_col in [
        ("Global MAD", "Global_MAD_Flag"),
        ("Type-Specific MAD", "Type_MAD_Flag")
    ]:

        department_results.append({
            "Store_Type": store_type,
            "Method": method,
            "Department_Weeks": len(group),
            "Flagged_Department_Weeks": int(
                group[flag_col].sum()
            ),
            "Flag_Rate_%": (
                100 * group[flag_col].mean()
            )
        })

department_comparison = pd.DataFrame(
    department_results
)

print("\nDEPARTMENT FLAG RATES BY STORE TYPE")
display(
    department_comparison.sort_values(
        ["Store_Type", "Method"]
    ).round(2)
)

# --------------------------------------------------
# 5. Aggregate department flags to store-week
# --------------------------------------------------

def aggregate_store_week(flag_column, method_name):

    result = (
        test.assign(
            Department_Flag=test[flag_column].astype(int)
        )
        .groupby(
            ["Store", "Date", "Type"],
            as_index=False
        )
        .agg(
            Actual_Sales=("Weekly_Sales", "sum"),
            Expected_Sales=("Prediction", "sum"),
            Department_Count=("Dept", "nunique"),
            Flagged_Departments=("Department_Flag", "sum")
        )
    )

    result["Flagged_Dept_Rate"] = (
        result["Flagged_Departments"]
        / result["Department_Count"]
    )

    result["Net_Sales_Gap"] = (
        result["Expected_Sales"]
        - result["Actual_Sales"]
    )

    result["Positive_Net_Gap"] = (
        result["Net_Sales_Gap"].clip(lower=0)
    )

    # Same store-week eligibility rule for both
    result["Eligible"] = (
        (result["Flagged_Dept_Rate"] >= 0.10)
        & (result["Net_Sales_Gap"] > 0)
    )

    result["Method"] = method_name

    return result

global_sw = aggregate_store_week(
    "Global_MAD_Flag", "Global MAD"
)

type_sw = aggregate_store_week(
    "Type_MAD_Flag", "Type-Specific MAD"
)

assert len(global_sw) == len(type_sw)

print("\nStore-weeks evaluated:", len(global_sw))

# --------------------------------------------------
# 6. Compare store-week eligibility by type
# --------------------------------------------------

combined_sw = pd.concat(
    [global_sw, type_sw],
    ignore_index=True
)

workload = (
    combined_sw.groupby(["Method", "Type"])
    .agg(
        Store_Weeks=("Store", "size"),
        Eligible_Store_Weeks=("Eligible", "sum")
    )
    .reset_index()
)

workload["Eligibility_Rate_%"] = (
    100 * workload["Eligible_Store_Weeks"]
    / workload["Store_Weeks"]
)

print("\nSTORE-WEEK REVIEW WORKLOAD BY TYPE")
display(workload.round(2))

# --------------------------------------------------
# 7. Compare top-20 review queues
# --------------------------------------------------

total_positive_gap = (
    global_sw["Positive_Net_Gap"].sum()
)

summary_rows = []
top20_frames = []

for method, data in [
    ("Global MAD", global_sw),
    ("Type-Specific MAD", type_sw)
]:

    eligible = (
        data[data["Eligible"]]
        .sort_values(
            ["Positive_Net_Gap", "Store", "Date"],
            ascending=[False, True, True]
        )
        .copy()
    )

    top20 = eligible.head(20).copy()

    top20["Priority_Rank"] = range(
        1, len(top20) + 1
    )

    top20_frames.append(top20)

    selected_gap = top20["Positive_Net_Gap"].sum()

    summary_rows.append({
        "Method": method,
        "Eligible_Store_Weeks": len(eligible),
        "Top20_Cases": len(top20),
        "Top20_Modeled_Gap_$": selected_gap,
        "Top20_Gap_Coverage_%": (
            100 * selected_gap / total_positive_gap
            if total_positive_gap > 0 else np.nan
        ),
        "Type_A_Top20": int(
            (top20["Type"] == "A").sum()
        ),
        "Type_B_Top20": int(
            (top20["Type"] == "B").sum()
        ),
        "Type_C_Top20": int(
            (top20["Type"] == "C").sum()
        )
    })

summary = pd.DataFrame(summary_rows)

top20_all = pd.concat(
    top20_frames,
    ignore_index=True
)

print("\nTOP-20 EXCEPTION COMPARISON")
display(summary.round(2))

# --------------------------------------------------
# 8. Measure agreement between the two MAD methods
# --------------------------------------------------

global_keys = set(zip(
    global_sw.loc[global_sw["Eligible"], "Store"],
    global_sw.loc[global_sw["Eligible"], "Date"]
))

type_keys = set(zip(
    type_sw.loc[type_sw["Eligible"], "Store"],
    type_sw.loc[type_sw["Eligible"], "Date"]
))

agreement = pd.DataFrame([{
    "Flagged_By_Both": len(global_keys & type_keys),
    "Global_Only": len(global_keys - type_keys),
    "Type_Specific_Only": len(type_keys - global_keys),
    "Jaccard_Similarity": (
        len(global_keys & type_keys)
        / len(global_keys | type_keys)
        if global_keys | type_keys else np.nan
    )
}])

print("\nSTORE-WEEK AGREEMENT")
display(agreement.round(3))

# --------------------------------------------------
# 9. Check the three presentation examples
# --------------------------------------------------

selected_cases = pd.DataFrame({
    "Store": [27, 10, 28],
    "Date": pd.to_datetime([
        "2012-09-14",
        "2012-08-31",
        "2012-09-14"
    ])
})

case_check = (
    selected_cases
    .merge(
        global_sw[
            ["Store", "Date", "Eligible",
             "Flagged_Dept_Rate", "Positive_Net_Gap"]
        ].rename(columns={
            "Eligible": "Global_Eligible",
            "Flagged_Dept_Rate": "Global_Flag_Rate",
            "Positive_Net_Gap": "Modeled_Gap"
        }),
        on=["Store", "Date"],
        how="left"
    )
    .merge(
        type_sw[
            ["Store", "Date", "Eligible",
             "Flagged_Dept_Rate"]
        ].rename(columns={
            "Eligible": "Type_Eligible",
            "Flagged_Dept_Rate": "Type_Flag_Rate"
        }),
        on=["Store", "Date"],
        how="left"
    )
)

print("\nTHREE PRESENTATION CASES")
display(case_check.round(3))

# --------------------------------------------------
# 10. Save results
# --------------------------------------------------

output_file = (
    BASE / "Walmart_Global_vs_Type_MAD.xlsx"
)

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    global_stats.to_frame("Global").T.to_excel(
        writer,
        sheet_name="Global_Threshold",
        index=False
    )

    type_stats.reset_index().to_excel(
        writer,
        sheet_name="Type_Thresholds",
        index=False
    )

    department_comparison.to_excel(
        writer,
        sheet_name="Department_Flag_Rates",
        index=False
    )

    workload.to_excel(
        writer,
        sheet_name="Store_Type_Workload",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Top20_Comparison",
        index=False
    )

    top20_all.to_excel(
        writer,
        sheet_name="Top20_Cases",
        index=False
    )

    agreement.to_excel(
        writer,
        sheet_name="MAD_Agreement",
        index=False
    )

    case_check.to_excel(
        writer,
        sheet_name="Three_Cases",
        index=False
    )

    combined_sw.to_excel(
        writer,
        sheet_name="All_Store_Weeks",
        index=False
    )

print("\nAnalysis completed.")
print("Excel saved:", output_file)


Historical calibration rows: 94243
Final test rows: 42402

GLOBAL MAD THRESHOLD


,Historical_N,Median_Residual,MAD,Negative_Threshold
Global,94243.0,-107.52,666.5,-3071.97



TYPE-SPECIFIC MAD THRESHOLDS


,Historical_N,Median_Residual,MAD,Negative_Threshold
Type,,,,
A,48386.0,-165.58,864.53,-4010.84
B,36671.0,-108.42,592.95,-2745.75
C,9186.0,-25.73,181.29,-832.07



DEPARTMENT FLAG RATES BY STORE TYPE


,Store_Type,Method,Department_Weeks,Flagged_Department_Weeks,Flag_Rate_%
0,A,Global MAD,21791,1727,7.93
1,A,Type-Specific MAD,21791,1099,5.04
2,B,Global MAD,16426,738,4.49
3,B,Type-Specific MAD,16426,895,5.45
4,C,Global MAD,4185,69,1.65
5,C,Type-Specific MAD,4185,390,9.32


... (5 outputs left out)

,Flagged_By_Both,Global_Only,Type_Specific_Only,Jaccard_Similarity
0,83,44,37,0.506



THREE PRESENTATION CASES


,Store,Date,Global_Eligible,Global_Flag_Rate,Modeled_Gap,Type_Eligible,Type_Flag_Rate
0,27,2012-09-14,True,0.362,240020.310,True,0.290
1,10,2012-08-31,True,0.353,156812.001,True,0.397
2,28,2012-09-14,True,0.145,116898.603,True,0.116



Analysis completed.
Excel saved: /content/drive/MyDrive/Walmart Recruiting Sales Data/Walmart_Global_vs_Type_MAD.xlsx


In [29]:
import pandas as pd

for name, value in globals().copy().items():
    if isinstance(value, pd.DataFrame):
        print(f"{name}: {value.shape}")
        print("Columns:", value.columns.tolist())
        print("-" * 60)

valid: (42400, 16)
Columns: ['Store', 'Dept', 'Date', 'Type', 'Actual_Sales', 'Expected_Sales', 'Dollar_Shortfall', 'Shortfall_Pct', 'Flag_GlobalMAD', 'Flag_TypeMAD', 'Flag_Fixed10', 'Flag_P90', 'Flag_P95', 'Residual_Check', 'Positive_Shortfall_Check', 'Raw_Shortfall_Check']
------------------------------------------------------------
results: (675, 9)
Columns: ['Store', 'Date', 'Type', 'Weekly_Sales', 'MA4_Residual_Pct', 'IF_A_Flag', 'IF_A_Score', 'IF_B_Flag', 'IF_B_Score']
------------------------------------------------------------
by_type: (15, 7)
Columns: ['Model', 'Type', 'Observations', 'MAE_$', 'RMSE_$', 'WAPE_%', 'Mean_Bias_$']
------------------------------------------------------------
group: (4185, 11)
Columns: ['Store', 'Dept', 'Date', 'Type', 'Weekly_Sales', 'Prediction', 'Residual', 'Global_MAD_Flag', 'Type_Threshold', 'Type_MAD_Flag', 'Positive_Gap']
------------------------------------------------------------
dept_summary: (3, 5)
Columns: ['Type', 'Department_Weeks', '

In [35]:

import pandas as pd
import numpy as np
import json
import zipfile
import shutil
from pathlib import Path
from google.colab import files

# ============================================================
# WALMART RAG v4 — FINAL ET + GLOBAL MAD JSON EXPORT
#
# EXISTING NOTEBOOK DATAFRAMES:
# global_sw = Latest calibrated Global MAD store-week results
# results   = Isolation Forest A/B results
# test      = Latest calibrated department-week results
# features  = Markdown, holiday and economic context
# sw        = Store-week historical sales
#
# OUTPUT:
# 1 master JSON containing all eligible cases
# 1 JSON file per case
# 1 ZIP containing all exported files
# ============================================================


# ============================================================
# 1. OUTPUT FOLDERS
# ============================================================

OUTPUT = Path("walmart_rag_v4_final")
CASES_FOLDER = OUTPUT / "cases"

# Remove old export files to avoid stale JSON records.
# This only removes the folder generated by this script.

if OUTPUT.exists():
    shutil.rmtree(OUTPUT)

CASES_FOLDER.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. USE EXISTING DATAFRAMES
# ============================================================

base = global_sw.copy()
if_results = results.copy()
departments = test.copy()
context = features.copy()
sales_history = sw.copy()

for frame in [
    base,
    if_results,
    departments,
    context,
    sales_history
]:
    frame["Date"] = pd.to_datetime(
        frame["Date"]
    ).dt.strftime("%Y-%m-%d")


# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def bool_value(value):

    if pd.isna(value):
        raise ValueError("Missing required Boolean value")

    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    normalized = str(value).strip().lower()

    if normalized in ("true", "1", "1.0"):
        return True

    if normalized in ("false", "0", "0.0"):
        return False

    raise ValueError(f"Invalid Boolean: {value}")


def safe_number(value):

    if pd.isna(value):
        return None

    value = float(value)

    if not np.isfinite(value):
        return None

    return value


def safe_int(value):

    if pd.isna(value):
        return None

    return int(value)


# ============================================================
# 4. PREPARE CORRECT DEPARTMENT DATA
# ============================================================

# IMPORTANT:
# Use the latest "test" DataFrame, not the older "valid".
# Your audit showed 0 mismatches across 675 store-weeks.

departments = departments.rename(columns={
    "Weekly_Sales": "Actual_Sales",
    "Prediction": "Expected_Sales",
    "Global_MAD_Flag": "Flag_GlobalMAD"
})

departments["Dollar_Shortfall"] = (
    departments["Expected_Sales"]
    - departments["Actual_Sales"]
)

departments["Shortfall_Pct"] = np.where(
    departments["Expected_Sales"] != 0,
    100 * departments["Dollar_Shortfall"]
    / departments["Expected_Sales"],
    np.nan
)

departments["Flag_GlobalMAD"] = (
    departments["Flag_GlobalMAD"].map(bool_value)
)


# ============================================================
# 5. VALIDATE UNIQUE KEYS
# ============================================================

for name, frame in [
    ("global_sw", base),
    ("results", if_results),
    ("features", context),
    ("sw", sales_history)
]:

    if frame.duplicated(["Store", "Date"]).any():

        raise ValueError(
            f"Duplicate Store-Date records in {name}"
        )

if departments.duplicated(
    ["Store", "Dept", "Date"]
).any():

    raise ValueError(
        "Duplicate Store-Dept-Date in department data"
    )


# ============================================================
# 6. MERGE GLOBAL MAD WITH IF-A AND IF-B
# ============================================================

combined = base.merge(
    if_results[
        [
            "Store",
            "Date",
            "IF_A_Flag",
            "IF_B_Flag",
            "IF_A_Score",
            "IF_B_Score",
            "MA4_Residual_Pct"
        ]
    ],
    on=["Store", "Date"],
    how="left",
    validate="one_to_one",
    indicator=True
)

if not combined["_merge"].eq("both").all():

    raise ValueError(
        "Some Global MAD store-weeks have no IF results"
    )

combined = combined.drop(columns="_merge")


# ============================================================
# 7. ADD MARKDOWN AND ECONOMIC CONTEXT
# ============================================================

combined = combined.merge(
    context[
        [
            "Store",
            "Date",
            "IsHoliday",
            "Temperature",
            "Fuel_Price",
            "CPI",
            "Unemployment",
            "MarkDown1",
            "MarkDown2",
            "MarkDown3",
            "MarkDown4",
            "MarkDown5"
        ]
    ],
    on=["Store", "Date"],
    how="left",
    validate="one_to_one",
    indicator=True
)

if not combined["_merge"].eq("both").all():

    raise ValueError(
        "Missing store-week contextual data"
    )

combined = combined.drop(columns="_merge")


# ============================================================
# 8. CONVERT BOOLEAN FLAGS
# ============================================================

for column in [
    "Eligible",
    "IF_A_Flag",
    "IF_B_Flag"
]:

    combined[column] = combined[column].map(
        bool_value
    )


# ============================================================
# 9. SELECT ALL GLOBAL MAD ELIGIBLE CASES
# ============================================================

eligible = combined.loc[
    combined["Eligible"]
    & (combined["Net_Sales_Gap"] > 0)
].copy()

eligible = eligible.sort_values(
    [
        "Net_Sales_Gap",
        "Store",
        "Date"
    ],
    ascending=[
        False,
        True,
        True
    ]
).reset_index(drop=True)

eligible["Priority_Rank"] = np.arange(
    1,
    len(eligible) + 1
)

print("=" * 60)
print("GLOBAL MAD ELIGIBLE CASES:", len(eligible))
print("IF-A SUPPORT:", int(eligible["IF_A_Flag"].sum()))
print("IF-B SUPPORT:", int(eligible["IF_B_Flag"].sum()))
print("=" * 60)

# Expected from your latest notebook:
# 127 eligible cases

assert len(eligible) == 127, (
    "Eligible count differs from your validated "
    "127-case experiment. Check the source DataFrames."
)


# ============================================================
# 10. CREATE DEPARTMENT LOOKUP
# ============================================================

department_lookup = {
    (int(store), date): group.copy()

    for (store, date), group
    in departments.groupby(["Store", "Date"])
}


def get_department_evidence(store, date):

    rows = department_lookup.get(
        (store, date)
    )

    if rows is None:
        raise ValueError(
            f"Missing department data: Store {store}, {date}"
        )

    records = []

    for _, row in rows.iterrows():

        records.append({

            "department": safe_int(row["Dept"]),

            "actual_sales": safe_number(
                row["Actual_Sales"]
            ),

            "expected_sales": safe_number(
                row["Expected_Sales"]
            ),

            "modeled_shortfall": safe_number(
                row["Dollar_Shortfall"]
            ),

            "shortfall_pct": safe_number(
                row["Shortfall_Pct"]
            ),

            "global_mad_flag": bool(
                row["Flag_GlobalMAD"]
            )

        })

    records = sorted(
        records,
        key=lambda x: (
            x["modeled_shortfall"]
            if x["modeled_shortfall"] is not None
            else float("-inf")
        ),
        reverse=True
    )

    return records


# ============================================================
# 11. HISTORICAL SALES LOOKUP
# ============================================================

history_lookup = {

    int(store): group.sort_values("Date").copy()

    for store, group
    in sales_history.groupby("Store")
}


def get_history(store, date):

    history = history_lookup.get(store)

    if history is None:
        raise ValueError(
            f"Missing historical sales for Store {store}"
        )

    previous = history.loc[
        history["Date"] < date,
        ["Date", "Weekly_Sales"]
    ].tail(8)

    observations = []

    for _, row in previous.iterrows():

        observations.append({
            "date": row["Date"],
            "weekly_sales": safe_number(
                row["Weekly_Sales"]
            )
        })

    if previous.empty:

        average = None
        deviation = None

    else:

        average = float(
            previous["Weekly_Sales"].mean()
        )

        current = history.loc[
            history["Date"] == date,
            "Weekly_Sales"
        ]

        if len(current) > 0 and average != 0:

            deviation = (
                float(current.iloc[0]) / average - 1
            ) * 100

        else:
            deviation = None

    return {
        "previous_8_weeks": observations,
        "previous_8_week_average": average,
        "deviation_from_previous_8w_pct": deviation
    }


# ============================================================
# 12. CREATE JSON CASE
# ============================================================

def create_case(row):

    store = int(row["Store"])
    date = row["Date"]

    if_a = bool(row["IF_A_Flag"])
    if_b = bool(row["IF_B_Flag"])

    # Markdown fields

    markdowns = {
        f"MarkDown{i}": safe_number(
            row[f"MarkDown{i}"]
        )
        for i in range(1, 6)
    }

    recorded = [
        value
        for value in markdowns.values()
        if value is not None
    ]

    # Descriptive model agreement

    if if_a and if_b:
        agreement = "ET + IF-A + IF-B"

    elif if_a:
        agreement = "ET + IF-A"

    elif if_b:
        agreement = "ET + IF-B"

    else:
        agreement = "ET only"

    dept_evidence = get_department_evidence(
        store,
        date
    )

    history = get_history(
        store,
        date
    )

    return {

        "schema_version": "4.0-draft",

        "case_id": f"Store_{store}_{date}",

        "priority_rank": int(
            row["Priority_Rank"]
        ),

        # ----------------------------------------------------
        # SOURCE
        # ----------------------------------------------------

        "source": {

            "project": (
                "Walmart Markdown Performance "
                "Exception Investigation"
            ),

            "primary_model": (
                "Extra Trees + Global MAD"
            ),

            "selection_method": "ET-first",

            "isolation_forest_role": "Supporting",

            "export_status": "draft",

            "test_period": (
                "2012-07-20 to 2012-10-26"
            ),

            "department_source": (
                "Latest calibrated test DataFrame"
            )
        },

        # ----------------------------------------------------
        # STORE-WEEK
        # ----------------------------------------------------

        "store_week": {

            "store": store,

            "date": date,

            "type": str(
                row["Type"]
            )

        },

        # ----------------------------------------------------
        # SALES
        # ----------------------------------------------------

        "sales_performance": {

            "actual_sales": safe_number(
                row["Actual_Sales"]
            ),

            "expected_sales": safe_number(
                row["Expected_Sales"]
            ),

            "net_modeled_gap": safe_number(
                row["Net_Sales_Gap"]
            ),

            # Numeric amount, not a Boolean

            "positive_net_gap": safe_number(
                row["Positive_Net_Gap"]
            ),

            "has_positive_net_gap": bool(
                row["Net_Sales_Gap"] > 0
            )

        },

        # ----------------------------------------------------
        # ML DETECTION SIGNALS
        # ----------------------------------------------------

        "model_signals": {

            "et_global_mad_flag": True,

            "if_a_flag": if_a,

            "if_b_flag": if_b,

            "if_a_score": safe_number(
                row["IF_A_Score"]
            ),

            "if_b_score": safe_number(
                row["IF_B_Score"]
            ),

            "ma4_residual_pct": safe_number(
                row["MA4_Residual_Pct"]
            ),

            "model_agreement": agreement,

            "if_required_for_eligibility": False

        },

        # ----------------------------------------------------
        # DEPARTMENT EVIDENCE
        # ----------------------------------------------------

        "department_summary": {

            "department_count": safe_int(
                row["Department_Count"]
            ),

            "flagged_departments": safe_int(
                row["Flagged_Departments"]
            ),

            "flagged_department_rate": safe_number(
                row["Flagged_Dept_Rate"]
            ),

            "departments": dept_evidence

        },

        # ----------------------------------------------------
        # SALES HISTORY
        # ----------------------------------------------------

        "historical_sales": history,

        # ----------------------------------------------------
        # CONTEXT
        # ----------------------------------------------------

        "business_context": {

            "is_holiday": (
                None
                if pd.isna(row["IsHoliday"])
                else bool_value(
                    row["IsHoliday"]
                )
            ),

            "temperature": safe_number(
                row["Temperature"]
            ),

            "fuel_price": safe_number(
                row["Fuel_Price"]
            ),

            "cpi": safe_number(
                row["CPI"]
            ),

            "unemployment": safe_number(
                row["Unemployment"]
            ),

            "markdown_fields": markdowns,

            "markdown_data_available": (
                len(recorded) > 0
            ),

            "total_recorded_markdown": (
                float(sum(recorded))
                if recorded
                else None
            )

        },

        # ----------------------------------------------------
        # INVESTIGATION
        # ----------------------------------------------------

        "investigation": {

            "status": "Needs analyst review",

            "recommended_next_step": (
                "Review department contributors, "
                "historical trends, markdown records, "
                "and relevant contextual evidence."
            ),

            "causality_verified": False,

            "modeled_gap_is_confirmed_loss": False,

            "human_review_required": True

        }
    }


# ============================================================
# 13. CREATE ALL CASES
# ============================================================

full_cases = [
    create_case(row)
    for _, row in eligible.iterrows()
]


# ============================================================
# 14. VERIFY DEPARTMENT FLAGS
# ============================================================

mismatches = []

for case in full_cases:

    summary = case["department_summary"]

    actual_flags = sum(
        dept["global_mad_flag"]
        for dept in summary["departments"]
    )

    expected_flags = summary["flagged_departments"]

    if actual_flags != expected_flags:

        mismatches.append({
            "case_id": case["case_id"],
            "expected": expected_flags,
            "actual": actual_flags
        })

print("\nDEPARTMENT VERIFICATION")
print("Exported cases:", len(full_cases))
print("Flag count mismatches:", len(mismatches))

if mismatches:
    print("First mismatches:", mismatches[:5])

assert len(mismatches) == 0, (
    "Department flags do not match "
    "store-week summaries."
)


# ============================================================
# 15. VERIFY STORE-WEEK SALES TOTALS
# ============================================================

sales_mismatches = []

for case in full_cases:

    dept_records = case["department_summary"]["departments"]

    department_actual = sum(
        d["actual_sales"]
        for d in dept_records
        if d["actual_sales"] is not None
    )

    store_actual = case["sales_performance"]["actual_sales"]

    if store_actual is not None and not np.isclose(
        department_actual,
        store_actual,
        atol=0.05,
        rtol=1e-8
    ):

        sales_mismatches.append({
            "case_id": case["case_id"],
            "store_actual": store_actual,
            "department_actual": department_actual
        })

print("Sales aggregation mismatches:", len(sales_mismatches))

# A sales mismatch may reflect department eligibility
# or a different aggregation universe.
# We report it rather than silently changing figures.

if sales_mismatches:
    print("First mismatches:", sales_mismatches[:3])


# ============================================================
# 16. EXPORT INDIVIDUAL CASES
# ============================================================

for case in full_cases:

    rank = case["priority_rank"]

    filename = (
        f"rank_{rank:03d}_"
        f"{case['case_id']}.json"
    )

    with open(
        CASES_FOLDER / filename,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            case,
            f,
            indent=2,
            ensure_ascii=False,
            allow_nan=False
        )


# ============================================================
# 17. EXPORT MASTER JSON
# ============================================================

master_path = (
    OUTPUT / "all_eligible_ET_GlobalMAD.json"
)

with open(
    master_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        full_cases,
        f,
        indent=2,
        ensure_ascii=False,
        allow_nan=False
    )


# ============================================================
# 18. CREATE ZIP FILE
# ============================================================

zip_path = Path(
    "walmart_rag_v4_final_127_cases.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as z:

    z.write(
        master_path,
        arcname=master_path.name
    )

    for path in sorted(
        CASES_FOLDER.glob("*.json")
    ):

        z.write(
            path,
            arcname=f"cases/{path.name}"
        )


# ============================================================
# 19. VERIFY YOUR PRESENTATION CASES
# ============================================================

print("\nTHREE PRESENTATION CASES")
print("-" * 60)

targets = [
    (27, "2012-09-14"),
    (10, "2012-08-31"),
    (28, "2012-09-14")
]

for store, date in targets:

    match = next(
        (
            case
            for case in full_cases
            if case["store_week"]["store"] == store
            and case["store_week"]["date"] == date
        ),
        None
    )

    if match:

        print(
            f"Store {store} | {date} | "
            f"Rank {match['priority_rank']} | "
            f"{match['model_signals']['model_agreement']}"
        )

    else:

        print(
            f"Store {store} | {date} | NOT ELIGIBLE"
        )


# ============================================================
# 20. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("WALMART RAG v4 EXPORT COMPLETE")
print("=" * 60)

print("Total cases:", len(full_cases))
print(
    "Individual JSON files:",
    len(list(CASES_FOLDER.glob("*.json")))
)
print("Department mismatches:", len(mismatches))
print("Sales aggregation mismatches:", len(sales_mismatches))
print("ZIP:", zip_path)

if sales_mismatches:
    print(
        "\nWARNING: Review sales aggregation mismatches "
        "before sharing this package."
    )
else:
    print("\nAll department and sales checks passed.")

files.download(str(zip_path))


GLOBAL MAD ELIGIBLE CASES: 127
IF-A SUPPORT: 27
IF-B SUPPORT: 15

DEPARTMENT VERIFICATION
Exported cases: 127
Flag count mismatches: 0
Sales aggregation mismatches: 0

THREE PRESENTATION CASES
------------------------------------------------------------
Store 27 | 2012-09-14 | Rank 1 | ET only
Store 10 | 2012-08-31 | Rank 7 | ET + IF-A
Store 28 | 2012-09-14 | Rank 19 | ET + IF-A + IF-B

WALMART RAG v4 EXPORT COMPLETE
Total cases: 127
Individual JSON files: 127
Department mismatches: 0
Sales aggregation mismatches: 0
ZIP: walmart_rag_v4_final_127_cases.zip

All department and sales checks passed.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>